# Board2Notes · standalone teammate pipeline

**This notebook contains the pipeline code. Share the `.ipynb` itself.**

Photo → detect writing → selected correction → crop text lines → PaddleOCR → text/JSON for the LLM.

Use **Run All** after editing the settings cell. No project `.py` files or `preprocessing/` folder are required. Advanced implementation cells below are folded where the notebook viewer supports it; expand them to inspect the code. Notebook 08 remains unchanged.

## 1. Environment

Select the working Python 3.12 OCR kernel. Standard installed packages and model weights are still required. The first use of a pretrained model may download its weights. No training runs.

For a fresh CPU environment, run these once in a notebook cell, then restart the kernel:

```python
%pip install paddlepaddle==3.3.1 paddleocr==3.7.0 paddlex==3.7.2 opencv-contrib-python==4.10.0.84 numpy==2.3.5 Pillow==12.3.0 pillow-heif==1.8.1
```

Keep an existing verified PaddlePaddle GPU installation when using NVIDIA; the CPU install command above is for a fresh CPU environment. No additional project code needs to be copied.

In [ ]:
from pathlib import Path
import importlib.util
import os
import sys

os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")
missing = [name for name in ("paddle", "paddleocr", "cv2", "PIL") if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(f"Missing packages: {missing}. Install the environment packages above and restart the kernel.")
print("Python:", sys.version.split()[0])
print("Working folder:", Path.cwd())

### Embedded implementation · run these cells once

The following cells define the pipeline **inside this notebook**. They create isolated in-memory namespaces so helper functions do not overwrite one another. They do not write or load project Python files. Model imports still use installed PaddleOCR normally.

In [ ]:
import hashlib
import types
from IPython.core.magic import register_cell_magic

_embedded_package = types.ModuleType("b2n_team_embedded")
_embedded_package.__path__ = []  # no filesystem search for implementation modules
_embedded_package.__file__ = "<embedded notebook package>"
sys.modules[_embedded_package.__name__] = _embedded_package

def _load_team_module(name, source):
    fullname = "b2n_team_embedded." + name
    module = types.ModuleType(fullname)
    module.__package__ = "b2n_team_embedded"
    module.__file__ = "<embedded " + name + ">"
    module.__source__ = source
    module.__source_sha256__ = hashlib.sha256(source.encode()).hexdigest()
    sys.modules[fullname] = module
    setattr(_embedded_package, name, module)
    exec(compile(source, module.__file__, "exec"), module.__dict__)
    return module

@register_cell_magic
def team_module(line, cell):
    _load_team_module(line.strip(), cell)

In [ ]:
%%team_module text_lines
"""Text-line detection, cutting and enlarging (hurdle slide 8 "Zoom in / Enlarge").

Taken from Assignment/section8_auto_rectify_cells.ipynb section 9 (smart_ocr steps 1-4) with the globals removed.

  Detector.detect(img)      -> text-line boxes, the same way the app finds them (smart_ocr):
                               1. quick detection on the photo shrunk to 1600 px;
                               2. if the writing is smaller than DET_MIN_H px as the detector saw it,
                                  re-detect each text region at full resolution.
  group_into_lines(boxes)   -> boxes on the same line joined (the detector sometimes splits a line at a big gap)
  reading_order(boxes)      -> top-to-bottom, left-to-right
  quad_crop(img, box)       -> the line cut along its own (possibly tilted) box, like PaddleOCR does
  prepare_for_reader(crop)  -> lines shorter than REC_MIN_H px are enlarged to TARGET_H with bicubic + sharpen
                               (the reader works at 48 px; we tested ESPCN/TPGSR: no better, see
                               docs/SUPER_RESOLUTION_RUNS.md)

PaddleOCR is imported only when a Detector is created, so the geometry helpers work without it.
"""
from __future__ import annotations

import cv2
import numpy as np

DET_MIN_H = 20        # writing smaller than this (px, as the 1600 px detector sees it) -> re-detect at full resolution
REC_MIN_H = 48        # lines shorter than this (px) are enlarged before reading (recognizer input height = 48 px)
TARGET_H = 64         # ... to about this height
MAX_SCALE = 4.0
QUICK_LIMIT = 1600
REGION_MAX = 4000     # PaddleOCR shrinks anything larger than this

# the detection preset of paddleocr_baseline.ipynb ("baseline"), used everywhere in the project
DET_ARGS = dict(thresh=0.20, box_thresh=0.40, unclip_ratio=2.00)


# ---------------------------------------------------------------- box helpers
def as_poly(p):
    return np.asarray(p, float).reshape(-1, 2)


def box_area(p):
    return float(cv2.contourArea(np.float32(as_poly(p))))


def line_height(poly):
    # height of a (roughly horizontal) text-line box: mean of its two short sides
    p = as_poly(poly)
    if len(p) != 4:
        return float(np.ptp(p[:, 1]))
    return float((np.linalg.norm(p[0] - p[3]) + np.linalg.norm(p[1] - p[2])) / 2)


def line_width(poly):
    p = as_poly(poly)
    if len(p) != 4:
        return float(np.ptp(p[:, 0]))
    return float((np.linalg.norm(p[0] - p[1]) + np.linalg.norm(p[3] - p[2])) / 2)


def order_tl_tr_br_bl(p):
    p = np.float32(as_poly(p))
    if len(p) != 4:
        p = np.float32(cv2.boxPoints(cv2.minAreaRect(p)))
    s, d = p.sum(1), np.diff(p, axis=1).ravel()
    return np.float32([p[s.argmin()], p[d.argmin()], p[s.argmax()], p[d.argmax()]])


def merge_boxes(polys, overlap=0.5):
    # drop boxes that lie mostly inside a bigger one (same line found twice in overlapping regions)
    polys = sorted((np.float32(as_poly(p)) for p in polys), key=lambda p: -box_area(p))
    kept = []
    for a in polys:
        area = max(box_area(a), 1)
        if all(cv2.intersectConvexConvex(a, k)[0] / area <= overlap for k in kept):
            kept.append(a)
    return kept


def text_regions(polys, img_shape, grow=1.5, margin=0.6):
    # group nearby boxes into clusters; one full-resolution region (with margin) per cluster
    h, w = img_shape[:2]
    rects = []
    for p in polys:
        p = as_poly(p); lh = line_height(p)
        rects.append([*(p.min(0) - grow * lh), *(p.max(0) + grow * lh), lh])
    parent = list(range(len(rects)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]; i = parent[i]
        return i
    for i in range(len(rects)):
        for j in range(i + 1, len(rects)):
            a, b = rects[i], rects[j]
            if a[0] < b[2] and b[0] < a[2] and a[1] < b[3] and b[1] < a[3]:
                parent[find(i)] = find(j)
    groups = {}
    for i in range(len(rects)):
        groups.setdefault(find(i), []).append(i)
    regions = []
    for idx in groups.values():
        pts = np.vstack([as_poly(polys[i]) for i in idx])
        lh = np.median([rects[i][4] for i in idx])
        (x0, y0), (x1, y1) = pts.min(0), pts.max(0)
        mx, my = max(margin * (x1 - x0), 4 * lh), max(margin * (y1 - y0), 4 * lh)
        regions.append([int(max(0, x0 - mx)), int(max(0, y0 - my)), int(min(w, x1 + mx)), int(min(h, y1 + my))])
    merged = True
    while merged:
        merged = False
        for i in range(len(regions)):
            for j in range(i + 1, len(regions)):
                a, b = regions[i], regions[j]
                if a[0] < b[2] and b[0] < a[2] and a[1] < b[3] and b[1] < a[3]:
                    regions[i] = [min(a[0], b[0]), min(a[1], b[1]), max(a[2], b[2]), max(a[3], b[3])]
                    regions.pop(j); merged = True
                    break
            if merged:
                break
    return [tuple(r) for r in regions]


def on_mask(polys, mask):
    # boxes whose centre is on the board (all boxes if there is no board mask or none are on it)
    if mask is None:
        return list(polys)
    h, w = mask.shape
    keep = [p for p in polys if mask[int(np.clip(as_poly(p)[:, 1].mean(), 0, h - 1)),
                                     int(np.clip(as_poly(p)[:, 0].mean(), 0, w - 1))] > 0]
    return keep or list(polys)


# ---------------------------------------------------------------- lines: grouping and order
def group_into_lines(polys, gap_factor=1.2, min_v_overlap=0.5):
    """Join boxes that sit on the same text line (vertical overlap >= min_v_overlap of the smaller box, and a
    horizontal gap below gap_factor x line height). Returns a list of 4-point boxes (minAreaRect of the parts)
    and, for each, the indices of the boxes it was made from."""
    P = [order_tl_tr_br_bl(p) for p in polys]
    n = len(P)
    parent = list(range(n))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]; i = parent[i]
        return i
    ext = [(p[:, 0].min(), p[:, 1].min(), p[:, 0].max(), p[:, 1].max(), line_height(p)) for p in P]
    for i in range(n):
        for j in range(i + 1, n):
            ax0, ay0, ax1, ay1, ah = ext[i]
            bx0, by0, bx1, by1, bh = ext[j]
            v_ov = min(ay1, by1) - max(ay0, by0)
            if v_ov < min_v_overlap * min(ay1 - ay0, by1 - by0):
                continue
            if max(ah, bh) > 2.0 * min(ah, bh):           # very different sizes: a title next to body text
                continue
            gap = max(bx0 - ax1, ax0 - bx1)
            if gap < gap_factor * max(ah, bh):
                parent[find(i)] = find(j)
    groups = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)
    out = []
    for idx in groups.values():
        if len(idx) == 1:
            out.append((P[idx[0]], idx))
        else:
            pts = np.vstack([P[i] for i in idx])
            out.append((order_tl_tr_br_bl(cv2.boxPoints(cv2.minAreaRect(np.float32(pts)))), sorted(idx)))
    return [o[0] for o in out], [o[1] for o in out]


def reading_order(polys):
    """Indices of the boxes top-to-bottom, then left-to-right within a row (a row = boxes whose vertical centres
    are within half a line height of each other)."""
    if not polys:
        return []
    c = [(as_poly(p)[:, 0].mean(), as_poly(p)[:, 1].mean(), line_height(p)) for p in polys]
    idx = sorted(range(len(polys)), key=lambda i: c[i][1])
    rows, cur = [], [idx[0]]
    for i in idx[1:]:
        ref = np.mean([c[k][1] for k in cur]); lh = np.median([c[k][2] for k in cur])
        if abs(c[i][1] - ref) < 0.5 * lh:
            cur.append(i)
        else:
            rows.append(cur); cur = [i]
    rows.append(cur)
    return [i for r in rows for i in sorted(r, key=lambda k: c[k][0])]


# ---------------------------------------------------------------- cutting and enlarging
def quad_crop(img, poly, pad=0.0):
    """Cut a text line along its own (possibly tilted) box, like PaddleOCR does internally.
    pad: grow the box by this fraction of its height on every side first (0 = exactly the box)."""
    p = order_tl_tr_br_bl(poly)
    if pad:
        lh = line_height(p)
        ux = (p[1] - p[0]) / max(np.linalg.norm(p[1] - p[0]), 1e-6)     # along the line
        uy = (p[3] - p[0]) / max(np.linalg.norm(p[3] - p[0]), 1e-6)     # across the line
        p = (p + np.float32([(-ux - uy), (ux - uy), (ux + uy), (-ux + uy)]) * pad * lh).astype(np.float32)
    w = int(round(max(np.linalg.norm(p[0] - p[1]), np.linalg.norm(p[3] - p[2]))))
    h = int(round(max(np.linalg.norm(p[0] - p[3]), np.linalg.norm(p[1] - p[2]))))
    M = cv2.getPerspectiveTransform(p, np.float32([[0, 0], [w, 0], [w, h], [0, h]]))
    return cv2.warpPerspective(img, M, (max(w, 1), max(h, 1)), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)


def enlarge(img, s, method="bicubic+sharpen"):
    # enlarge an image by factor s (no learned super-resolution: it did not help, see SUPER_RESOLUTION_RUNS.md)
    if s <= 1.05 or method == "none":
        return img
    size = (int(round(img.shape[1] * s)), int(round(img.shape[0] * s)))
    if method == "bicubic":
        return cv2.resize(img, size, interpolation=cv2.INTER_CUBIC)
    if method == "lanczos":
        return cv2.resize(img, size, interpolation=cv2.INTER_LANCZOS4)
    if method == "bicubic+sharpen":
        up = cv2.resize(img, size, interpolation=cv2.INTER_CUBIC)
        blur = cv2.GaussianBlur(up, (0, 0), sigmaX=0.8 * s)
        return cv2.addWeighted(up, 1.6, blur, -0.6, 0)           # unsharp mask: strengthens stroke edges
    raise ValueError(f"unknown enlargement method {method!r}")


def prepare_for_reader(crop, height=None, method="bicubic+sharpen"):
    """What the app feeds the reader: lines shorter than REC_MIN_H are enlarged to ~TARGET_H, others unchanged.
    height: the line's box height (px); the crop height if not given."""
    lh = float(height if height is not None else crop.shape[0])
    if lh >= REC_MIN_H:
        return crop
    return enlarge(crop, float(np.clip(TARGET_H / max(lh, 1), 1, MAX_SCALE)), method)


def ink_rows(crop):
    """Number of text rows in a line crop (1 = one line). Horizontal ink profile after dividing out the board
    background; bands shorter than 12 % of the crop or 40 % of the tallest band (underlines, the dot of an i, an
    arrow tip) do not count. Checked by eye on 64 crops (23 Oct): it found the two-line crops and none of the
    single lines with underlines or arrows."""
    g = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY).astype(np.float32) if crop.ndim == 3 else crop.astype(np.float32)
    h = g.shape[0]
    if h < 8:
        return 1
    bg = cv2.GaussianBlur(g, (0, 0), max(3.0, h / 2))
    ink = (np.clip(g / np.maximum(bg, 1), 0, 1.5) < 0.8).astype(np.float32)
    k = max(1, int(round(h / 20)))
    prof = np.convolve(ink.mean(1), np.ones(k) / k, mode="same")
    if prof.max() <= 0:
        return 0
    on = list(prof > 0.3 * prof.max()) + [False]
    bands, start = [], None
    for y, v in enumerate(on):
        if v and start is None:
            start = y
        if not v and start is not None:
            bands.append([start, y]); start = None
    merged = []
    for b in bands:
        if merged and b[0] - merged[-1][1] < 0.06 * h:
            merged[-1][1] = b[1]
        else:
            merged.append(b)
    big = [b for b in merged if b[1] - b[0] >= 0.12 * h]
    if not big:
        return 1
    tall = max(b[1] - b[0] for b in big)
    return len([b for b in big if b[1] - b[0] >= 0.4 * tall])


# ---------------------------------------------------------------- the detector (PaddleOCR, lazy)
class Detector:
    """PaddleOCR text detection with the app's settings (PP-OCRv5_mobile_det, the baseline preset).
    The mobile detector was ~6x faster than PP-OCRv6_medium_det on a 1600 px tile and found the same lines
    (section8 notebook, section 9.6)."""

    def __init__(self, model_name="PP-OCRv5_mobile_det", det_args=None):
        import os
        os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")
        from paddleocr import TextDetection
        self.model_name = model_name
        self.model = TextDetection(model_name=model_name)
        self.args = dict(DET_ARGS, **(det_args or {}))

    def boxes(self, img_bgr, limit=None, unclip=None):
        # text boxes only. limit=1600 shrinks the image to at most 1600 px; None = full resolution (<= REGION_MAX)
        # unclip: override of the box growth (2.0 in the baseline preset; large values join neighbouring lines)
        kw = dict(self.args, limit_side_len=limit or REGION_MAX, limit_type="max")
        if unclip is not None:
            kw["unclip_ratio"] = unclip
        return [as_poly(p) for p in self.model.predict(img_bgr, **kw)[0]["dt_polys"]]

    def detect(self, img_bgr, board_mask_fn=None, unclip=None):
        """smart_ocr steps 1-3. board_mask_fn(img, boxes) -> mask or None (board_geometry.segment_board_floodfill)
        keeps the work on the board. Returns (boxes, info)."""
        h, w = img_bgr.shape[:2]
        shrink = min(1.0, QUICK_LIMIT / max(h, w))
        info = dict(path="", writing_px_detector=None, regions=0)
        quick = self.boxes(img_bgr, limit=QUICK_LIMIT, unclip=unclip)
        if not quick:
            info["path"] = "nothing found in the quick pass"
            return [], info
        mask = board_mask_fn(img_bgr, quick) if board_mask_fn else None
        writing = on_mask(quick, mask)
        det_h = float(np.median([line_height(p) for p in writing])) * shrink
        info["writing_px_detector"] = round(det_h, 1)
        if det_h >= DET_MIN_H:
            info["path"] = f"text large enough ({det_h:.0f} px) -> quick boxes kept"
            return writing, info
        regions = text_regions(writing, img_bgr.shape)
        polys = []
        for x0, y0, x1, y1 in regions:
            polys += [p + [x0, y0] for p in self.boxes(img_bgr[y0:y1, x0:x1], unclip=unclip)]
        info["regions"] = len(regions)
        info["path"] = f"small text ({det_h:.0f} px) -> {len(regions)} region(s) re-detected at full resolution"
        return merge_boxes(polys) or writing, info


class Reader:
    """PaddleOCR text-line recognition (the app's reader, PP-OCRv6_medium_rec unless a model_dir is given)."""

    def __init__(self, model_name="PP-OCRv6_medium_rec", model_dir=None):
        import os
        os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")
        from paddleocr import TextRecognition
        self.model = TextRecognition(model_name=model_name, **({"model_dir": str(model_dir)} if model_dir else {}))

    def read(self, images, batch=16):
        out = []
        for i in range(0, len(images), batch):
            out += [(r["rec_text"], float(r["rec_score"])) for r in self.model.predict(images[i:i + batch])]
        return out

In [ ]:
%%team_module board_geometry
"""Perspective correction of a whiteboard photo (hurdle slide 7).

Taken from Assignment/section8_auto_rectify_cells.ipynb (sections 8.1-8.3) with the notebook globals removed:
the PaddleOCR text boxes are PASSED IN (from preprocessing.text_lines.Detector), so this module needs only
OpenCV + NumPy + Pillow and can be tested without PaddleOCR.

How it works (strict frame-evidence gate added after the notebook experiments):
  1. find the board: flood-fill from the writing, stopped by strong edges (frame, tray)       segment_board_floodfill
  2. find long straight lines (Hough + merging of collinear fragments)                          detect_long_lines
  3. pick two board edges that are parallel in reality, around the writing                      choose_plane_edges
  4. their vanishing point + the camera focal length (EXIF) give the board's rotation R;
     the homography K R K^-1 turns the board to face the camera                                  rotation_from_edges
  5. warp, keeping the resolution of the writing                                                fit_to_canvas_at

Changes from the notebook (the notebook rectifies around ONE target line; a dataset needs every line):
  * the close-up fallback (rectify the single largest text box) is OFF by default: it keeps one box only;
  * the edge pair is anchored at the median of all writing and must enclose >= MIN_INSIDE of the text boxes;
  * the result keeps all the writing on the board (not only the band between the two edges), and a warp is
    rejected if it would push text out of the image (MIN_TEXT_KEPT) or turns the board > MAX_BOARD_ANGLE.
  Failures found in the first order-check run (22 Oct): oct17_040 was cut down to the lone word "output";
  oct17_002 / oct17_015 were cut to a strip or over-warped. Without usable edges the photo stays unrectified.
"""
from __future__ import annotations

from pathlib import Path

import cv2
import numpy as np

MAX_SIDE = 4000          # max width/height of a rectified image (px)
DEFAULT_F35 = 24         # iPhone 14 Pro main camera (the EXIF of our PNGs says 24 mm); used if EXIF is missing

# ---------------------------------------------------------------- camera
def focal35_from_exif(path) -> float | None:
    """35 mm-equivalent focal length from EXIF (our PNGs keep the iPhone EXIF), or None."""
    try:
        from PIL import Image
        with Image.open(path) as im:
            f35 = im.getexif().get_ifd(0x8769).get(0xA405)
        return float(f35) if f35 else None
    except Exception:
        return None


def camera_matrix(w, h, f35=DEFAULT_F35):
    # K from the 35 mm-equivalent focal length (a 35 mm frame is 36 mm wide)
    f = f35 / 36.0 * max(w, h)
    return np.array([[f, 0, w / 2], [0, f, h / 2], [0, 0, 1.0]])


def line_through(p, q):
    # homogeneous line through two image points
    return np.cross([*p, 1.0], [*q, 1.0])


# ---------------------------------------------------------------- 1. board region (flood fill, no model)
FLOOD_WORK_SIDE = 1000   # long side (px) of the downscaled copy used for flood fill
FLOOD_DIFF = 3           # max grey-level step between neighbouring pixels (gradual lighting passes, hard edges stop)


def segment_board_floodfill(img_bgr, boxes):
    """Board mask (uint8 0/255, full res) grown from the writing, stopped by strong edges; None if no boxes."""
    h, w = img_bgr.shape[:2]
    if not boxes:
        return None
    s = FLOOD_WORK_SIDE / max(h, w)
    small = cv2.resize(img_bgr, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    gray = cv2.GaussianBlur(cv2.cvtColor(small, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    sh, sw = gray.shape

    centres = [b.mean(0) for b in boxes]
    k = min(range(len(boxes)), key=lambda i: np.hypot(centres[i][0] - w / 2, centres[i][1] - h / 2))
    x0, y0 = (boxes[k].min(0) * s).astype(int)
    x1, y1 = (boxes[k].max(0) * s).astype(int)

    barrier = cv2.dilate(cv2.Canny(gray, 20, 60), np.ones((3, 3), np.uint8))
    barrier[max(0, y0):y1, max(0, x0):x1] = 0
    ff_mask = np.zeros((sh + 2, sw + 2), np.uint8)
    ff_mask[1:-1, 1:-1] = (barrier > 0).astype(np.uint8)

    pad = max(4, int(0.3 * (y1 - y0)))
    seeds = [(x0 - pad, (y0 + y1) // 2), (x1 + pad, (y0 + y1) // 2), ((x0 + x1) // 2, y0 - pad),
             ((x0 + x1) // 2, y1 + pad), ((x0 + x1) // 2, (y0 + y1) // 2)]
    filled = ff_mask.copy()
    for sx, sy in seeds:
        sx, sy = int(np.clip(sx, 0, sw - 1)), int(np.clip(sy, 0, sh - 1))
        if filled[sy + 1, sx + 1] == 0:
            cv2.floodFill(gray.copy(), filled, (sx, sy), 255, FLOOD_DIFF, FLOOD_DIFF,
                          4 | cv2.FLOODFILL_MASK_ONLY | (2 << 8))
    region = (filled[1:-1, 1:-1] == 2).astype(np.uint8) * 255

    region = cv2.morphologyEx(region, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
    n, lab = cv2.connectedComponents(region)
    cy, cx = int(np.clip((y0 + y1) / 2, 0, sh - 1)), int(np.clip((x0 + x1) / 2, 0, sw - 1))
    keep = lab[cy, cx] if lab[cy, cx] else (np.bincount(lab.ravel())[1:].argmax() + 1 if n > 1 else 0)
    region = (lab == keep).astype(np.uint8) * 255
    cnts, _ = cv2.findContours(region, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    region = cv2.drawContours(np.zeros_like(region), cnts, -1, 255, cv2.FILLED)
    return cv2.resize(region, (w, h), interpolation=cv2.INTER_NEAREST)


BOUNDARY_BAND = 0.015    # a line counts as a board edge if it runs within this fraction of the long side of the mask outline
BOUNDARY_FRAC = 0.6      # ... for at least this fraction of its length


def keep_boundary_lines(lines, mask):
    # only lines that run along the outline of the board mask (the real board edges), not the photo border
    h, w = mask.shape
    s = 1000 / max(h, w)
    small = cv2.resize(mask, None, fx=s, fy=s, interpolation=cv2.INTER_NEAREST)
    outline = cv2.morphologyEx(small, cv2.MORPH_GRADIENT, np.ones((3, 3), np.uint8))
    outline[:2, :] = 0; outline[-2:, :] = 0; outline[:, :2] = 0; outline[:, -2:] = 0
    dist = cv2.distanceTransform(255 - outline, cv2.DIST_L2, 3)
    band = BOUNDARY_BAND * 1000
    kept = []
    for e in lines:
        t = np.linspace(0, 1, 40)
        xs = (e["p"][0] + t * (e["q"][0] - e["p"][0])) * s
        ys = (e["p"][1] + t * (e["q"][1] - e["p"][1])) * s
        ok = (xs >= 0) & (xs < small.shape[1]) & (ys >= 0) & (ys < small.shape[0])
        if ok.sum() < 5:
            continue
        if (dist[ys[ok].astype(int), xs[ok].astype(int)] < band).mean() >= BOUNDARY_FRAC:
            kept.append(e)
    return kept


# ---------------------------------------------------------------- 2-3. long lines and the two board edges
LINE_WORK_SIDE = 1600   # long side (px) of the downscaled copy used for line detection
MIN_LINE_FRAC = 0.12    # minimum merged line length, as a fraction of the long side
VERTICAL_TOL = 25       # degrees from image-vertical that still counts as a vertical line
MAX_GAP_FRAC = 0.05     # max gap between line fragments that are merged into one line
MIN_PAIR_FRAC = 0.15    # the two plane edges must each be at least this long (fraction of the long side)
MIN_REF_FRAC = 0.25     # a reference line must be at least this long


def detect_long_lines(img_bgr):
    # long straight lines in full-res coords, with nearly-collinear Hough fragments merged
    h, w = img_bgr.shape[:2]
    s = LINE_WORK_SIDE / max(h, w)
    g = cv2.cvtColor(cv2.resize(img_bgr, None, fx=s, fy=s, interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2GRAY)
    e = cv2.Canny(cv2.GaussianBlur(g, (5, 5), 0), 15, 45)
    segs = cv2.HoughLinesP(e, 1, np.pi / 720, 40, minLineLength=int(0.04 * LINE_WORK_SIDE), maxLineGap=12)
    if segs is None:
        return []
    segs = np.asarray(segs).reshape(-1, 4).astype(float) / s     # (N,1,4) in OpenCV 4, (N,4) in OpenCV 5
    diag = np.hypot(w, h)
    info = []
    for x1, y1, x2, y2 in segs:
        th = np.arctan2(y2 - y1, x2 - x1) % np.pi
        n = np.array([-np.sin(th), np.cos(th)])
        info.append((th, n @ [x1, y1], np.hypot(x2 - x1, y2 - y1), (x1, y1, x2, y2)))
    used, lines = [False] * len(info), []
    for i in np.argsort([-l for _, _, l, _ in info]):
        if used[i]:
            continue
        th0, r0 = info[i][0], info[i][1]
        group = []
        for j, (th, r, l, seg) in enumerate(info):
            dth = min(abs(th - th0), np.pi - abs(th - th0))
            r_adj = r if abs(th - th0) < np.pi / 2 else -r
            if not used[j] and dth < np.radians(1.5) and abs(r_adj - r0) < 0.005 * diag:
                used[j] = True
                group.append((l, seg))
        d = np.array([np.cos(th0), np.sin(th0)])
        spans = sorted((min(d @ seg[:2], d @ seg[2:]), max(d @ seg[:2], d @ seg[2:]), l, seg) for l, seg in group)
        runs, cur, end = [], [spans[0]], spans[0][1]
        for sp in spans[1:]:
            if sp[0] - end > MAX_GAP_FRAC * max(h, w):
                runs.append(cur); cur = []
            cur.append(sp); end = max(end, sp[1])
        runs.append(cur)
        best_run = max(runs, key=lambda r: sum(x[2] for x in r))
        support = sum(x[2] for x in best_run)
        if support < MIN_LINE_FRAC * max(h, w):
            continue
        pts = np.float32([p for *_, seg in best_run for p in (seg[:2], seg[2:])])
        vx, vy, x0, y0 = cv2.fitLine(pts, cv2.DIST_L2, 0, 0.01, 0.01).ravel()
        t = (pts - [x0, y0]) @ [vx, vy]
        p, q = (x0 + vx * t.min(), y0 + vy * t.min()), (x0 + vx * t.max(), y0 + vy * t.max())
        deg_from_vertical = np.degrees(np.arccos(abs(vy)))
        extent = float(np.hypot(q[0] - p[0], q[1] - p[1]))
        lines.append(dict(p=p, q=q, length=extent, support=support, kind="v" if deg_from_vertical < VERTICAL_TOL else "h"))
    return lines


def y_at_x(e, x):
    (x1, y1), (x2, y2) = e["p"], e["q"]
    return y1 + (y2 - y1) * (x - x1) / (x2 - x1)


def x_at_y(e, y):
    (x1, y1), (x2, y2) = e["p"], e["q"]
    return x1 + (x2 - x1) * (y - y1) / (y2 - y1)


def _mid(e):
    return ((e["p"][0] + e["q"][0]) / 2, (e["p"][1] + e["q"][1]) / 2)


def _cross_inside(a, b, w, h):
    # True if the two (extended) lines intersect inside the photo: real parallel board edges meet only outside it
    x, y, z = np.cross(line_through(a["p"], a["q"]), line_through(b["p"], b["q"]))
    return abs(z) > 1e-9 and 0 <= x / z <= w and 0 <= y / z <= h


MIN_INSIDE = 0.5        # a chosen edge pair must have at least this fraction of the writing between it


def choose_plane_edges(lines, boxes, mask, min_inside=None):
    """Pair of non-crossing board lines around the writing, both LONG and CLOSE to it.

    Changed from the notebook (which aims at ONE target line, the box nearest the photo centre):
      * anchor = the median centre of all writing on the board, not the box nearest the photo centre
        (on oct17_040 that box was a lone word, "output", and everything else was cropped away);
      * every candidate pair is ranked by the notebook's score, and the best one with at least
        `min_inside` of the text boxes between its two lines is taken (on close-ups, Hough lines along the
        handwriting itself were paired around one row of text).
    """
    min_inside = MIN_INSIDE if min_inside is None else min_inside   # read at call time, so a changed setting counts
    h, w = mask.shape
    grow = cv2.dilate(mask, np.ones((int(0.03 * max(h, w)),) * 2, np.uint8))
    on_board = lambda e: grow[int(np.clip(_mid(e)[1], 0, h - 1)), int(np.clip(_mid(e)[0], 0, w - 1))] > 0
    centres = [b.mean(0) for b in boxes]
    in_mask = [k for k, (x, y) in enumerate(centres) if grow[int(np.clip(y, 0, h - 1)), int(np.clip(x, 0, w - 1))] > 0]
    pool = in_mask or list(range(len(boxes)))
    ax_, ay_ = np.median(np.array([centres[k] for k in pool]), axis=0)
    for axis in ("h", "v"):                      # prefer top/bottom edges; fall back to left/right
        cand = [e for e in lines if e["kind"] == axis and on_board(e) and e["length"] >= MIN_PAIR_FRAC * max(h, w)]
        pos = (lambda e: y_at_x(e, ax_)) if axis == "h" else (lambda e: x_at_y(e, ay_))
        anchor = ay_ if axis == "h" else ax_
        before = [e for e in cand if pos(e) < anchor]
        after = [e for e in cand if pos(e) > anchor]
        scored = []
        for lo in before:
            for hi in after:
                if not _cross_inside(lo, hi, w, h):
                    scored.append((min(lo["length"], hi["length"]) / (pos(hi) - pos(lo)), lo, hi))
        for _, lo, hi in sorted(scored, key=lambda t: -t[0]):
            if axis == "h":
                inside = [k for k in pool if y_at_x(lo, centres[k][0]) < centres[k][1] < y_at_x(hi, centres[k][0])]
            else:
                inside = [k for k in pool if x_at_y(lo, centres[k][1]) < centres[k][0] < x_at_y(hi, centres[k][1])]
            if len(inside) < min_inside * len(pool):
                continue
            choice = dict(axis=axis, pair=(lo, hi), boxes=inside)
            other = [e for e in lines if e["kind"] != axis and e["length"] >= MIN_REF_FRAC * max(h, w)]
            choice["ref"] = max(other, key=lambda e: e["length"]) if other else None
            return choice
    return None


def refine_on_image(img_bgr, e):
    # snap a line to the strongest parallel edge in a band around it (full resolution)
    h, w = img_bgr.shape[:2]
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    band = 0.008 * max(h, w)
    p, q = np.array(e["p"]), np.array(e["q"])
    m = np.zeros_like(gray)
    cv2.line(m, tuple(map(int, p)), tuple(map(int, q)), 255, int(2 * band))
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 10, 40) & m
    ys, xs = np.nonzero(edges)
    if len(xs) < 50:
        return e
    d = np.float32(np.c_[xs, ys])
    vx, vy, x0, y0 = cv2.fitLine(d, cv2.DIST_HUBER, 0, 0.01, 0.01).ravel()
    proj = lambda r: (x0 + vx * ((r[0] - x0) * vx + (r[1] - y0) * vy), y0 + vy * ((r[0] - x0) * vx + (r[1] - y0) * vy))
    return dict(e, p=proj(p), q=proj(q))


# ---------------------------------------------------------------- 4-5. rotation, homography, warp
def rotation_from_edges(K, choice, img_shape):
    # board rotation R (rows: board x, y, normal in camera coords) from an opposite edge pair + optional reference
    h, w = img_shape[:2]
    Ki = np.linalg.inv(K)
    a, b = choice["pair"]
    vp = np.cross(line_through(a["p"], a["q"]), line_through(b["p"], b["q"]))
    d1 = Ki @ vp
    d1 /= np.linalg.norm(d1)
    ref = choice["ref"]
    if choice["axis"] == "h":
        l_ref = line_through(ref["p"], ref["q"]) if ref else line_through((K[0, 2], 0), (K[0, 2], h))
        d_h = d1 * np.sign(d1[0] or 1)
        d_v = np.cross(d_h, K.T @ l_ref)
    else:
        l_ref = line_through(ref["p"], ref["q"]) if ref else line_through((0, K[1, 2]), (w, K[1, 2]))
        d_v = d1 * np.sign(d1[1] or 1)
        d_h = np.cross(d_v, K.T @ l_ref)
        d_h /= np.linalg.norm(d_h)
        d_h *= np.sign(d_h[0] or 1)
    d_v /= np.linalg.norm(d_v)
    d_v *= np.sign(d_v[1] or 1)
    return np.stack([d_h, d_v, np.cross(d_h, d_v)])


def fit_to_canvas_at(img, H, roi, anchor, max_side=None, keep_pts=None):
    """Warp so that the resolution is preserved at `anchor` (the writing).
    keep_pts (new): points that must stay inside the output (all the writing). The canvas is centred on their
    bounding box instead of the anchor, and if they span more than max_side px the whole output is scaled down
    until they fit (the far end of a long, steeply angled board then loses some resolution instead of being
    cut off)."""
    max_side = MAX_SIDE if max_side is None else max_side
    x0, y0, x1, y1 = roi
    ax, ay, d = anchor[0], anchor[1], 5.0
    p = cv2.perspectiveTransform(np.float32([[[ax, ay]], [[ax + d, ay]], [[ax, ay + d]]]), H).reshape(-1, 2)
    J = np.stack([(p[1] - p[0]) / d, (p[2] - p[0]) / d], axis=1)
    s = 1.0 / np.sqrt(abs(np.linalg.det(J)))
    corners = cv2.perspectiveTransform(np.float32([[[x0, y0]], [[x1, y0]], [[x1, y1]], [[x0, y1]]]), H).reshape(-1, 2)
    if keep_pts is not None:
        kp = cv2.perspectiveTransform(np.float32(keep_pts).reshape(1, -1, 2), H)[0]
        lo, hi = kp.min(0), kp.max(0)
        pad = 0.05 * (hi - lo).max()
        span = float((hi - lo).max() + 2 * pad)
        s = min(s, max_side / max(span, 1e-6))
        centre = (lo + hi) / 2 * s
    else:
        centre = p[0] * s
    corners = corners * s
    bx0, by0 = np.maximum(corners.min(0), centre - max_side / 2)
    bx1, by1 = np.minimum(corners.max(0), centre + max_side / 2)
    H_out = np.array([[s, 0, -bx0], [0, s, -by0], [0, 0, 1.0]]) @ H
    out = cv2.warpPerspective(img, H_out, (max(1, int(bx1 - bx0)), max(1, int(by1 - by0))),
                              flags=cv2.INTER_CUBIC, borderValue=(255, 255, 255))
    return out, H_out


def crop_box(shape, pts, margin=0.6, min_px=150):
    """(x0, y0, x1, y1) around the points, grown by `margin` x their size (at least min_px)."""
    h, w = shape[:2]
    x0, y0 = pts.min(0); x1, y1 = pts.max(0)
    mx, my = max(margin * (x1 - x0), min_px), max(margin * (y1 - y0), min_px)
    return (int(max(0, x0 - mx)), int(max(0, y0 - my)), int(min(w, x1 + mx)), int(min(h, y1 + my)))


def crop_to_text(img, pts, margin=0.6):
    x0, y0, x1, y1 = crop_box(img.shape, pts, margin)
    return img[y0:y1, x0:x1]


def order_quad(points):
    # top-left, top-right, bottom-right, bottom-left
    pts = np.asarray(points, dtype=np.float32).reshape(-1, 2)
    if len(pts) != 4:
        pts = cv2.boxPoints(cv2.minAreaRect(pts)).astype(np.float32)
    ordered = np.zeros((4, 2), dtype=np.float32)
    s, d = pts.sum(axis=1), np.diff(pts, axis=1).ravel()
    ordered[0], ordered[2] = pts[np.argmin(s)], pts[np.argmax(s)]
    ordered[1], ordered[3] = pts[np.argmin(d)], pts[np.argmax(d)]
    return ordered


def rectify_from_text_box(img_bgr, boxes, margin=1.15):
    # close-up fallback of the notebook: rectify the largest text quadrilateral only
    valid = [np.asarray(b, dtype=np.float32).reshape(-1, 2) for b in boxes if np.asarray(b).size >= 8]
    if not valid:
        return None, None, None
    src = order_quad(max(valid, key=lambda b: abs(cv2.contourArea(b))))
    centre = src.mean(axis=0)
    src = centre + margin * (src - centre)
    tl, tr, br, bl = src
    out_w = max(2, int(round(max(np.linalg.norm(tr - tl), np.linalg.norm(br - bl)))))
    out_h = max(2, int(round(max(np.linalg.norm(bl - tl), np.linalg.norm(br - tr)))))
    dst = np.float32([[0, 0], [out_w - 1, 0], [out_w - 1, out_h - 1], [0, out_h - 1]])
    H = cv2.getPerspectiveTransform(src, dst)
    out = cv2.warpPerspective(img_bgr, H, (out_w, out_h), flags=cv2.INTER_CUBIC,
                              borderMode=cv2.BORDER_CONSTANT, borderValue=(255, 255, 255))
    return out, H, src


def between_mask(shape, choice):
    # pixels between the two chosen edges (the band of the board that holds the writing)
    h, w = shape[:2]
    a, b = choice["pair"]
    if choice["axis"] == "h":
        poly = [(0, y_at_x(a, 0)), (w, y_at_x(a, w)), (w, y_at_x(b, w)), (0, y_at_x(b, 0))]
    else:
        poly = [(x_at_y(a, 0), 0), (x_at_y(b, 0), 0), (x_at_y(b, h), h), (x_at_y(a, h), h)]
    m = np.zeros((h, w), np.uint8)
    cv2.fillPoly(m, [np.int32(np.clip(poly, -10 * max(h, w), 10 * max(h, w)))], 255)
    return m


MAX_BOARD_ANGLE = 75.0   # a board turned further than this is almost certainly a wrong edge pair
MIN_TEXT_KEPT = 0.9      # a rectification must keep at least this fraction of the text-box corners in the image


# Conservative, label-free proposal thresholds. Missing evidence means ABSTAIN,
# not 'frontal'. These are not a calibrated side-view classifier or accuracy claim.
SIDE_MIN_CONVERGENCE = 4.0   # degrees between top and bottom; roll alone stays parallel
SIDE_MIN_TAPER = 1.15        # board height must change >=15% over supported edge overlap
SIDE_MIN_EDGE = 0.25         # each observed frame segment / image long side
SIDE_MIN_OVERLAP = 0.25      # observed horizontal overlap / image width


def side_view_evidence(img_shape, boxes, choice, mask, f35):
    """Permit automatic yaw rectification only with supported board-frame evidence.

    Two mask-boundary top/bottom edges must converge, have substantial observed
    overlap and bracket ALL detected writing. A third boundary segment must join
    both as a real observed frame corner. Text boxes are NOT board corners.
    """
    metrics = {}
    def verdict(ok, reason):
        return dict(approved=bool(ok), reason=reason, metrics=metrics)
    if f35 is None or not np.isfinite(f35) or f35 <= 0:
        return verdict(False, "camera focal length unknown: abstain")
    if choice is None or choice.get("axis") != "h":
        return verdict(False, "need top and bottom board-frame edges: abstain")
    a, b = choice["pair"]
    ref = choice.get("ref")
    if ref is None:
        return verdict(False, "no observed board-frame corner/reference: abstain")
    if len(keep_boundary_lines([a, b, ref], mask)) != 3:
        return verdict(False, "edge/reference is not supported by board boundary: abstain")
    h, w = img_shape[:2]
    if min(float(e["length"]) for e in (a, b, ref)) < SIDE_MIN_EDGE * max(h, w):
        return verdict(False, "observed board-frame segments too short: abstain")
    dirs = [np.asarray(e["q"], float) - e["p"] for e in (a, b)]
    if min(abs(d[0]) for d in dirs) < 1e-6:
        return verdict(False, "invalid top/bottom directions: abstain")
    cosine = np.clip(abs(np.dot(*dirs)) / (np.linalg.norm(dirs[0]) * np.linalg.norm(dirs[1])), 0, 1)
    convergence = float(np.degrees(np.arccos(cosine)))
    metrics["convergence_deg"] = round(convergence, 3)
    if convergence < SIDE_MIN_CONVERGENCE:
        return verdict(False, "frame edges nearly parallel: no strong side-view evidence")
    left = max(min(e["p"][0], e["q"][0]) for e in (a, b))
    right = min(max(e["p"][0], e["q"][0]) for e in (a, b))
    overlap = (right - left) / w
    metrics["edge_overlap_fraction"] = round(float(overlap), 3)
    if overlap < SIDE_MIN_OVERLAP:
        return verdict(False, "too little observed edge overlap: abstain")
    gaps = [float(y_at_x(b, x) - y_at_x(a, x)) for x in (left, right)]
    if min(gaps) <= 0 or _cross_inside(a, b, w, h):
        return verdict(False, "frame edges cross or reverse: abstain")
    taper = max(gaps) / min(gaps)
    metrics["height_taper"] = round(taper, 3)
    if taper < SIDE_MIN_TAPER:
        return verdict(False, "insufficient frame taper: no strong side-view evidence")
    # The reference must actually span the two observed junctions, not just be
    # a vertical letter or a room line whose extension intersects somewhere.
    rp, rq = np.asarray(ref["p"], float), np.asarray(ref["q"], float)
    rd = rq - rp
    denom = float(rd @ rd)
    if denom < 1:
        return verdict(False, "invalid reference segment: abstain")
    for edge in (a, b):
        cross = np.cross(line_through(edge["p"], edge["q"]), line_through(rp, rq))
        if abs(cross[2]) < 1e-9:
            return verdict(False, "reference does not join both board edges: abstain")
        point = cross[:2] / cross[2]
        t = float((point - rp) @ rd / denom)
        ep, eq = np.asarray(edge["p"], float), np.asarray(edge["q"], float)
        ed = eq - ep
        u = float((point - ep) @ ed / max(float(ed @ ed), 1))
        if not (-0.1 <= t <= 1.1 and -0.1 <= u <= 1.1):
            return verdict(False, "frame junction lies beyond observed segments: abstain")
    text = np.vstack(boxes)
    inside = [(y_at_x(a, p[0]) - 0.01 * h <= p[1] <= y_at_x(b, p[0]) + 0.01 * h) for p in text]
    metrics["writing_inside_fraction"] = float(np.mean(inside))
    if not all(inside):
        return verdict(False, "frame pair does not bracket every detected text box: abstain")
    return verdict(True, "supported converging board frame: side-view candidate")


def auto_rectify(img_bgr, boxes, f35=None, text_box_fallback=False, strict=True, proposal_search=False):
    """Rectify the board the photo is aimed at.

    img_bgr : upright photo.   boxes : text boxes [(4,2) arrays, full-res coords] from the detector.
    f35     : 35 mm-equivalent focal length (focal35_from_exif(path)); DEFAULT_F35 if None.
    strict  : default True requires known focal length, converging supported frame
              edges and an observed corner before any warp. Missing/cropped frame
              means abstain, not a reliable declaration that the photo is frontal.
              False is an explicit legacy opt-out, not used by the normal pipeline.
    Returns info dict: status, rectified (or None), H (photo -> rectified), angle (board turn, deg),
    text_rect (the text points in the rectified image), mask, choice.
    """
    if proposal_search:
        return auto_rectify_supported(img_bgr, boxes, f35)
    h, w = img_bgr.shape[:2]
    info = dict(status="", rectified=None, H=None, angle=np.nan, mask=None, choice=None, text_rect=None)
    if not boxes:
        info["status"] = "no text found"
        return info
    info["mask"] = mask = segment_board_floodfill(img_bgr, boxes)
    if mask is None:
        info["status"] = "no whiteboard found"
        return info
    lines = detect_long_lines(img_bgr)
    boundary_lines = keep_boundary_lines(lines, mask)
    choice = choose_plane_edges(boundary_lines, boxes, mask)
    if choice is None and not strict:
        choice = choose_plane_edges(lines, boxes, mask)  # historical, explicit opt-out only
    if choice is None:
        if text_box_fallback and not strict:
            out, H, _ = rectify_from_text_box(img_bgr, boxes)
            if out is not None:
                info.update(rectified=out, H=H, status="largest text box only (no board edges)",
                            text_rect=cv2.perspectiveTransform(np.float32(np.vstack(boxes))[None], H)[0])
                return info
        info["status"] = "no board edges around the writing"
        return info
    info["choice"] = choice
    if strict:
        gate = side_view_evidence(img_bgr.shape, boxes, choice, mask, f35)
        info["side_view_gate"] = gate
        if not gate["approved"]:
            info["status"] = gate["reason"]
            return info
    # all the writing on the board (not only the part between the two edges): the crop must keep every line
    on = [b for b in boxes if mask[int(np.clip(b[:, 1].mean(), 0, h - 1)), int(np.clip(b[:, 0].mean(), 0, w - 1))] > 0]
    text = np.vstack(on or boxes)
    choice["pair"] = tuple(refine_on_image(img_bgr, e) for e in choice["pair"])
    if choice["ref"] is not None:
        choice["ref"] = refine_on_image(img_bgr, choice["ref"])
    info["choice"] = choice
    if strict:
        gate = side_view_evidence(img_bgr.shape, boxes, choice, mask, f35)
        info["side_view_gate"] = gate
        if not gate["approved"]:
            info["status"] = "refined edges rejected: " + gate["reason"]
            return info
    K = camera_matrix(w, h, f35 or DEFAULT_F35)
    R = rotation_from_edges(K, choice, img_bgr.shape)
    info["angle"] = float(np.degrees(np.arccos(min(1.0, abs(R[2, 2])))))
    if not np.isfinite(info["angle"]) or info["angle"] > MAX_BOARD_ANGLE:
        info["status"] = f"implausible board angle {info['angle']:.0f} deg: left unrectified"
        return info
    H = K @ R @ np.linalg.inv(K)
    # the plane = the segmented board (the notebook also cut it to the band between the two edges; that band can
    # leave out part of the writing now that the pair only needs MIN_INSIDE of it, so the band is not used)
    plane = cv2.dilate(mask, np.ones((int(0.01 * max(h, w)),) * 2, np.uint8))
    board_only = img_bgr.copy()
    board_only[plane == 0] = 255
    ys, xs = np.nonzero(plane)
    roi = (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))
    info["rectified"], info["H"] = fit_to_canvas_at(board_only, H, roi, anchor=text.mean(0), keep_pts=text)
    info["text_rect"] = cv2.perspectiveTransform(np.float32(text)[None], info["H"])[0]
    rh, rw = info["rectified"].shape[:2]
    tr = info["text_rect"]
    kept = float(np.mean((tr[:, 0] >= 0) & (tr[:, 0] < rw) & (tr[:, 1] >= 0) & (tr[:, 1] < rh)))
    required_kept = 1.0 if strict else MIN_TEXT_KEPT
    if kept < required_kept:
        info.update(rectified=None, H=None, text_rect=None,
                    status=f"rectification would cut off the writing ({kept:.0%} kept): left unrectified")
        return info
    info["status"] = (f"rectified from {'top/bottom' if choice['axis'] == 'h' else 'left/right'} edges + "
                      + ("reference line" if choice["ref"] else "no-roll assumption"))
    return info


# Revised, label-free proposal search. The historical strict route above stays
# available for reproduction. Neither manual annotations nor filenames are used.
FRAME_MIN_TARGET_AREA = 0.90
FRAME_MIN_VERTICAL = 0.12
TEXT_SMALL_HEIGHT = 16.0       # provisional ink-height proxy in native source pixels
TEXT_KEEP_SMALL = 0.95         # small writing must retain this share of ink height


def _intersection(a, b):
    p = np.cross(line_through(a['p'],a['q']),line_through(b['p'],b['q']))
    return None if abs(p[2])<1e-8 else p[:2]/p[2]


def _segment_parameter(e, point):
    p,q=np.asarray(e['p']),np.asarray(e['q']);d=q-p
    return float((point-p)@d/max(d@d,1))


def _frame_reference(top, bottom, ref, shape, edge_distance=None, edge_scale=1.):
    """Two observed joins, or one join plus an edge clipped at the image border.

    The latter is explicit geometric extrapolation, not an invented observed
    corner. Interior letters/room lines without a supported join cannot qualify.
    """
    h,w=shape[:2];points=[_intersection(e,ref) for e in (top,bottom)]
    if any(p is None for p in points):return False
    observed=[]
    for e,p in zip((top,bottom),points):
        on_edge=-.12<=_segment_parameter(e,p)<=1.12
        on_ref=-.12<=_segment_parameter(ref,p)<=1.12
        # A detector may stop a real frame segment early. Extend it only when
        # the intervening image pixels supply continuous edge evidence.
        if not on_edge and on_ref and edge_distance is not None:
            endpoint=min((np.asarray(e[k]) for k in ('p','q')),key=lambda q:np.linalg.norm(q-p))
            if np.linalg.norm(endpoint-p)<.15*max(h,w):
                samples=(endpoint+np.linspace(0,1,30)[:,None]*(p-endpoint))*edge_scale
                valid=(samples[:,0]>=0)&(samples[:,0]<edge_distance.shape[1])&(samples[:,1]>=0)&(samples[:,1]<edge_distance.shape[0])
                if valid.all():
                    ix=samples.astype(int);on_edge=np.mean(edge_distance[ix[:,1],ix[:,0]]<6)>=.8
        observed.append(on_edge and on_ref)
    if all(observed):return True
    if sum(observed)!=1:return False
    j=observed.index(False);p=points[j]
    outside=p[0]<0 or p[0]>=w or p[1]<0 or p[1]>=h
    near=np.min([np.linalg.norm(p-np.asarray(e[k])) for e in ((top,bottom)[j],ref) for k in ('p','q')])
    extrap=max(abs(_segment_parameter(ref,p)-.5)-.5,0)
    return bool(outside and near<.15*max(h,w) and extrap<.35)


def _box_panel_fraction(box, polygon):
    area=abs(cv2.contourArea(np.float32(box)))
    return float(cv2.intersectConvexConvex(np.float32(box),np.float32(polygon))[0]/max(area,1))


def detect_frame_lines(image):
    """Independent LSD proposals plus Hough; merge by endpoint distance to the
    seed line, retaining every supported run instead of discarding all but one.
    """
    h,w=image.shape[:2];scale=LINE_WORK_SIDE/max(h,w)
    gray=cv2.cvtColor(cv2.resize(image,None,fx=scale,fy=scale,interpolation=cv2.INTER_AREA),cv2.COLOR_BGR2GRAY)
    found=cv2.createLineSegmentDetector().detect(gray)[0]
    segments=[]
    if found is not None:
        for row in found.reshape(-1,4):
            p,q=row[:2]/scale,row[2:]/scale
            if np.linalg.norm(q-p)>=.04*max(h,w):segments.append((p,q))
    segments += [(np.asarray(e['p']),np.asarray(e['q'])) for e in detect_long_lines(image)]
    used=set();result=[];long=max(h,w)
    for i in sorted(range(len(segments)),key=lambda i:-np.linalg.norm(segments[i][1]-segments[i][0])):
        if i in used:continue
        p,q=segments[i];direction=(q-p)/np.linalg.norm(q-p);normal=np.array([-direction[1],direction[0]])
        group=[]
        for j,(a,b) in enumerate(segments):
            if j in used:continue
            d=(b-a)/max(np.linalg.norm(b-a),1)
            if abs(direction@d)<np.cos(np.radians(1.2)):continue
            if max(abs((a-p)@normal),abs((b-p)@normal))>.004*long:continue
            u,v=sorted((float(a@direction),float(b@direction)));group.append((u,v,j,a,b));used.add(j)
        runs=[];end=None
        for span in sorted(group,key=lambda s:s[0]):
            if end is None or span[0]-end>MAX_GAP_FRAC*long:runs.append([])
            runs[-1].append(span);end=max(end or span[1],span[1])
        for run in runs:
            extent=max(s[1] for s in run)-min(s[0] for s in run)
            if extent<MIN_LINE_FRAC*long:continue
            pts=np.float32([r[k] for r in run for k in (3,4)])
            vx,vy,x,y=cv2.fitLine(pts,cv2.DIST_HUBER,0,.01,.01).ravel();t=(pts-[x,y])@[vx,vy]
            a=np.array([x,y])+t.min()*np.array([vx,vy]);b=np.array([x,y])+t.max()*np.array([vx,vy])
            result.append(dict(p=a,q=b,length=float(np.linalg.norm(b-a)),kind='v' if abs(vy)>np.cos(np.radians(VERTICAL_TOL)) else 'h'))
    return result


def frame_proposals(image, boxes, mask, lines=None):
    """Search physical boundary triplets, rather than accepting the first pair
    and the globally longest vertical line. Permit a long internal panel joint
    only with an outline-supported lower edge and vertical frame reference.
    """
    h,w=image.shape[:2];long=max(h,w)
    lines=detect_frame_lines(image) if lines is None else lines
    boundary=keep_boundary_lines(lines,mask)
    edge_scale=1000/long
    small=cv2.resize(image,None,fx=edge_scale,fy=edge_scale,interpolation=cv2.INTER_AREA)
    edges=cv2.Canny(cv2.cvtColor(small,cv2.COLOR_BGR2GRAY),20,60)
    distance=cv2.distanceTransform(255-edges,cv2.DIST_L2,3)
    boundary_ids={id(e) for e in boundary}
    grown=cv2.dilate(mask,np.ones((max(3,int(.015*long)),)*2,np.uint8))
    def on_board(e):
        points=np.asarray(e['p'])+np.linspace(0,1,30)[:,None]*(np.asarray(e['q'])-e['p'])
        points=np.clip(points,[0,0],[w-1,h-1]).astype(int)
        return np.mean(grown[points[:,1],points[:,0]]>0)>=.60
    horizontals=[e for e in lines if e['kind']=='h' and e['length']>=SIDE_MIN_EDGE*long and on_board(e)]
    verticals=[e for e in lines if e['kind']=='v' and e['length']>=FRAME_MIN_VERTICAL*long]
    areas=np.array([abs(cv2.contourArea(np.float32(b))) for b in boxes]);total=max(areas.sum(),1)
    ax=float(np.median(np.vstack(boxes)[:,0]));proposals=[]
    for top in horizontals:
        for bottom in horizontals:
            if top is bottom or y_at_x(top,ax)>=y_at_x(bottom,ax):continue
            if _cross_inside(top,bottom,w,h):continue
            if id(bottom) not in boundary_ids:continue
            # Only long grey physical segments may act as an internal joint.
            for edge in (top,bottom):
                ts=np.linspace(.02,.98,60);ps=np.asarray(edge['p'])+ts[:,None]*(np.asarray(edge['q'])-edge['p'])
                ps=np.clip(ps,[0,0],[w-1,h-1]).astype(int)
                sat=cv2.cvtColor(image[ps[:,1],ps[:,0]][None],cv2.COLOR_BGR2HSV)[0,:,1]
                if np.mean(sat>70)>.25:break
            else:
                d1=np.asarray(top['q'])-top['p'];d2=np.asarray(bottom['q'])-bottom['p']
                angle=np.degrees(np.arccos(np.clip(abs(d1@d2)/(np.linalg.norm(d1)*np.linalg.norm(d2)),0,1)))
                if angle<SIDE_MIN_CONVERGENCE:continue
                left=max(min(top['p'][0],top['q'][0]),min(bottom['p'][0],bottom['q'][0]))
                right=min(max(top['p'][0],top['q'][0]),max(bottom['p'][0],bottom['q'][0]))
                if right-left<SIDE_MIN_OVERLAP*w:continue
                gaps=[y_at_x(bottom,x)-y_at_x(top,x) for x in (left,right)]
                if min(gaps)<=0 or max(gaps)/min(gaps)<SIDE_MIN_TAPER:continue
                refs=[]
                for r in verticals:
                    if not _frame_reference(top,bottom,r,image.shape,distance,edge_scale):continue
                    a,b=_intersection(top,r),_intersection(bottom,r)
                    clipped=dict(r,p=a,q=b,length=float(np.linalg.norm(b-a)))
                    if keep_boundary_lines([clipped],mask):refs.append(r)
                if not refs:continue
                # Enumerate left/right support independently; don't assume the
                # longest room vertical is the board side.
                for ref in refs:
                    xt=float(x_at_y(ref,float(np.median(np.vstack(boxes)[:,1]))))
                    candidates_left=[r for r in refs if x_at_y(r,h/2)<ax]
                    candidates_right=[r for r in refs if x_at_y(r,h/2)>ax]
                    le=max(candidates_left,key=lambda r:x_at_y(r,h/2),default=None)
                    re=min(candidates_right,key=lambda r:x_at_y(r,h/2),default=None)
                    corners=[_intersection(top,le) if le else [0,y_at_x(top,0)],
                             _intersection(top,re) if re else [w-1,y_at_x(top,w-1)],
                             _intersection(bottom,re) if re else [w-1,y_at_x(bottom,w-1)],
                             _intersection(bottom,le) if le else [0,y_at_x(bottom,0)]]
                    if any(p is None for p in corners):continue
                    polygon=np.float32(corners)
                    if not np.isfinite(polygon).all() or not cv2.isContourConvex(polygon):continue
                    inside=np.array([_box_panel_fraction(b,polygon)>=.90 for b in boxes])
                    area_kept=float(areas[inside].sum()/total)
                    if inside.sum()<3 or area_kept<FRAME_MIN_TARGET_AREA:continue
                    # A panel must agree with the flood-fill estimate. A room
                    # rectangle is not validated merely by containing text.
                    canvas=np.zeros((h,w),np.uint8);cv2.fillConvexPoly(canvas,np.int32(polygon),255)
                    overlap=float(np.count_nonzero((canvas>0)&(mask>0))/max(np.count_nonzero(canvas),1))
                    if overlap<.55:continue
                    rank=area_kept+min(top['length'],bottom['length'])/long+.2*(le is not None and re is not None)+.1*overlap+.35*(id(top) in boundary_ids)
                    proposals.append(dict(axis='h',pair=(top,bottom),ref=ref,polygon=polygon,
                        target_indices=np.flatnonzero(inside).tolist(),rank=float(rank),
                        evidence=dict(convergence_deg=float(angle),height_taper=float(max(gaps)/min(gaps)),
                            retained_box_area=area_kept,mask_overlap=overlap,observed_sides=int(le is not None)+int(re is not None))))
    proposals.sort(key=lambda p:-p['rank'])
    unique=[]
    for p in proposals:
        if not any(np.max(np.abs(p['polygon']-q['polygon']))<.015*long for q in unique):unique.append(p)
    return unique[:8]


def ink_height_proxy(image, box):
    """Connected ink-component height; a readability proxy, not character truth."""
    q=np.float32(box);width=max(8,int(max(np.linalg.norm(q[1]-q[0]),np.linalg.norm(q[2]-q[3]))))
    height=max(8,int(max(np.linalg.norm(q[3]-q[0]),np.linalg.norm(q[2]-q[1]))))
    s=min(1.,900/width,200/height);cw,ch=max(8,int(width*s)),max(8,int(height*s))
    m=cv2.getPerspectiveTransform(q,np.float32([[0,0],[cw-1,0],[cw-1,ch-1],[0,ch-1]]))
    crop=cv2.warpPerspective(image,m,(cw,ch),borderValue=(255,255,255))
    gray=cv2.cvtColor(crop,cv2.COLOR_BGR2GRAY).astype(np.float32)
    bg=cv2.GaussianBlur(gray,(0,0),max(3,ch/2));ink=(gray/np.maximum(bg,1)<.80).astype(np.uint8)
    _,_,stats,_=cv2.connectedComponentsWithStats(ink)
    heights=[r[cv2.CC_STAT_HEIGHT] for r in stats[1:] if r[cv2.CC_STAT_AREA]>=3 and .15*ch<=r[cv2.CC_STAT_HEIGHT]<=.98*ch]
    return float(np.percentile(heights,75)/s) if heights else None


def preservation_gate(image, boxes, H, shape):
    rows=[];required_boost=1.
    for box in boxes:
        b=np.float32(box);t=cv2.perspectiveTransform(b[None],H)[0]
        old=(np.linalg.norm(b[0]-b[3])+np.linalg.norm(b[1]-b[2]))/2
        new=(np.linalg.norm(t[0]-t[3])+np.linalg.norm(t[1]-t[2]))/2
        source=ink_height_proxy(image,b)
        unknown=source is None
        # No reliable glyph estimate: retain the original box height instead
        # of treating an uncertain/faint line as permission to shrink it.
        if unknown:source=float(old)
        target=source*new/max(old,1);floor=(source if unknown else min(source,TEXT_SMALL_HEIGHT))*TEXT_KEEP_SMALL
        required_boost=max(required_boost,floor/max(target,1e-6))
        rows.append(dict(source_ink_px=None if unknown else source,proposed_ink_px=float(target),required_ink_px=float(floor),
                         box_height_fallback=unknown,height_scale=float(new/max(old,1))))
    fits=required_boost*max(shape[:2])<=MAX_SIDE
    return dict(approved=bool(fits),reason='ink-height floor retained' if fits else 'readability would exceed output canvas limit',
                canvas_scale=float(required_boost),rows=rows)


def fit_panel_canvas(source, H, polygon, points):
    """Bound the observed panel itself, not its axis-aligned source rectangle.
    Bounding-rectangle corners can lie far outside a slanted board and near the
    homography horizon, wasting the canvas on an enormous white triangle.
    """
    h,w=source.shape[:2];frame=np.float32([[0,0],[w-1,0],[w-1,h-1],[0,h-1]])
    area,visible=cv2.intersectConvexConvex(np.float32(polygon),frame)
    if area<=0 or visible is None:raise ValueError('No observed panel inside photo')
    transformed=cv2.perspectiveTransform(np.float32(np.vstack([visible.reshape(-1,2),points]))[None],H)[0]
    if not np.isfinite(transformed).all():raise ValueError('Non-finite panel projection')
    anchor=points.mean(0);rays=np.float32([[anchor,anchor+[5,0],anchor+[0,5]]])
    projected=cv2.perspectiveTransform(rays,H)[0]
    jacobian=np.stack([(projected[1]-projected[0])/5,(projected[2]-projected[0])/5],axis=1)
    scale=1/max(np.sqrt(abs(np.linalg.det(jacobian))),1e-6)
    lo,hi=transformed.min(0),transformed.max(0);pad=max(5.,.02*float((hi-lo).max()))
    lo-=pad;hi+=pad;scale=min(scale,(MAX_SIDE-2)/max(float((hi-lo).max()),1))
    output=np.array([[scale,0,-lo[0]*scale],[0,scale,-lo[1]*scale],[0,0,1.]])@H
    size=np.maximum(1,np.ceil((hi-lo)*scale).astype(int))
    image=cv2.warpPerspective(source,output,tuple(size),flags=cv2.INTER_CUBIC,borderValue=(255,255,255))
    return image,output


def auto_rectify_supported(image, boxes, f35):
    info=dict(status='no supported frame proposal',rectified=None,H=None,angle=np.nan,mask=None,choice=None,text_rect=None)
    if not boxes or f35 is None or not np.isfinite(f35) or f35<=0:
        info['status']='text or camera information unavailable: abstain';return info
    mask=segment_board_floodfill(image,boxes);info['mask']=mask
    if mask is None:return info
    proposals=frame_proposals(image,boxes,mask)
    info['proposal_count']=len(proposals);rejections=[]
    for choice in proposals:
        k=camera_matrix(image.shape[1],image.shape[0],f35)
        r=rotation_from_edges(k,choice,image.shape)
        angle=float(np.degrees(np.arccos(np.clip(abs(r[2,2]),0,1))))
        if not np.isfinite(r).all() or not np.isfinite(angle) or not 0<=angle<=MAX_BOARD_ANGLE:continue
        target=[boxes[i] for i in choice['target_indices']];points=np.vstack(target)
        plane=np.zeros(image.shape[:2],np.uint8);cv2.fillConvexPoly(plane,np.int32(choice['polygon']),255)
        plane=cv2.dilate(plane,np.ones((7,7),np.uint8));source=image.copy();source[plane==0]=255
        result,H=fit_panel_canvas(source,k@r@np.linalg.inv(k),choice['polygon'],points)
        gate=preservation_gate(image,target,H,result.shape)
        if not gate['approved']:rejections.append(gate['reason']);continue
        boost=gate['canvas_scale']
        if boost>1.001:
            H=np.diag([boost,boost,1.])@H
            result=cv2.warpPerspective(source,H,(int(np.ceil(result.shape[1]*boost)),int(np.ceil(result.shape[0]*boost))),flags=cv2.INTER_CUBIC,borderValue=(255,255,255))
        tp=cv2.perspectiveTransform(np.float32(points)[None],H)[0]
        if not ((tp>=0).all() and (tp[:,0]<result.shape[1]).all() and (tp[:,1]<result.shape[0]).all()):
            rejections.append('writing outside output canvas');continue
        info.update(rectified=result,H=H,angle=angle,text_rect=tp,choice=choice,
            target_indices=choice['target_indices'],excluded_indices=[i for i in range(len(boxes)) if i not in choice['target_indices']],
            readability_gate=gate,side_view_gate=dict(approved=True,reason='supported physical frame/joint proposal',metrics=choice['evidence']),
            status='supported frame and ink-height preservation passed')
        return info
    info['rejections']=rejections
    return info

In [ ]:
%%team_module fix_lighting
#!/usr/bin/env python3
"""
1. Install dependencies


pip install opencv-python numpy
2. Run it on an image


cd ~/Downloads

python fix_lighting.py in.jpg out.jpg
## python fix_lighting.py example1.png example1_fixed.png ##
This writes the lighting-corrected image to out.jpg.

3. Other useful modes
# Just see the metrics without editing the file:
python fix_lighting.py in.jpg --analyze

# Gentler correction (useful before cropping/perspective-warp steps):
python fix_lighting.py in.jpg out.jpg --strength light

# Visual before/field/after comparison strip:
python fix_lighting.py in.jpg out.jpg --debug debug.jpg
###
fix_lighting.py - Lighting normalisation for document / whiteboard photos.

One preprocessing problem, one module. Handles the four faces of "bad lighting":

    1. globally too dim        -> illumination field scaled to a target paper
                                  level (with denoise first, since lifting a
                                  dark frame lifts its sensor noise too)
    2. globally too bright     -> same field, plus a contrast stretch to undo
                                  the wash toward white
    3. uneven illumination     -> flat-fielding: divide the estimated
                                  illumination field out. This is the big win
                                  for documents - a shadow across the page, a
                                  lamp gradient, and the ceiling-light streaks
                                  on a glossy whiteboard are all the same
                                  thing: a smooth multiplicative field.
    4. specular glare          -> detected, suppressed where recoverable, and
                                  REPORTED where not. Blown-out pixels are
                                  physically unrecoverable from a single
                                  frame; we never inpaint text back in,
                                  because invented strokes are worse than a
                                  missing region once an LLM reads the output.

Public API (the shape the future dispatcher needs):

    analyze(img)                 -> dict of metrics + issues + severity, no edit
    fix(img, strength=...)       -> (fixed_img, report_dict)

`strength`:
    'light'   gentle, partial normalisation, meant to run BEFORE geometry
              (corner detection / perspective warp) so page-vs-background
              edges are not flattened away
    'strong'  full normalisation, meant to run AFTER the warp when the frame
              contains only the document
    'auto'    'strong', with the correction amount scaled by measured severity

Classical OpenCV only. No model downloads, no GPU, deterministic.
Input is never binarised; colour in -> colour out, grey in -> grey out.

CLI:
    python fix_lighting.py in.jpg out.jpg
    python fix_lighting.py in.jpg out.jpg --strength light
    python fix_lighting.py in.jpg --analyze
    python fix_lighting.py in.jpg out.jpg --debug debug.jpg
"""

from __future__ import annotations

import argparse
import json
import os
import sys
from typing import Any, Dict, Tuple

import cv2
import numpy as np

__all__ = ["analyze", "fix", "LightingReport", "debug_strip"]

# ----------------------------------------------------------------------------
# Tunables. Grouped here so the graded harness can sweep them.
# ----------------------------------------------------------------------------

ANALYSIS_MAX_DIM = 1024      # illumination is estimated on a downscaled copy
BG_KERNEL_FRAC = 0.055       # background kernel as a fraction of min(h, w)
BG_KERNEL_MIN = 15

TARGET_PAPER = 236.0         # where the background should land after flat-field
MAX_GAIN = 14.0              # cap on per-pixel lift (noise guard)
MIN_GAIN = 0.2

DIM_BG = 105.0               # paper level below this -> "dim"
BRIGHT_BG = 251.0            # paper level above this -> "bright" (clipping)
UNEVEN_RATIO = 0.13          # illumination spread / level above this -> "uneven"
GLARE_FRAC = 0.002           # >0.2% specular pixels -> "glare"
LOW_INK_CONTRAST = 45.0      # mean ink depth below this -> "low_contrast"

NOISE_SIGMA_DENOISE = 2.6    # estimated sigma above this -> denoise before lift
NLM_MAX_PIXELS = 4_000_000   # above this, use bilateral instead of NL-means

CLAHE_CLIP_STRONG = 1.7
CLAHE_CLIP_LIGHT = 1.1
CLAHE_TILES = 8              # upper bound; actual grid is sized to the image
CLAHE_SKIP_INK_DEPTH = 85.0  # strokes already this deep -> CLAHE adds only
                             # halos and tile texture, so skip it

SPEC_ABOVE_BG = 14           # how far above the local paper level counts as a
                             # highlight rather than just bright paper
SPEC_MIN_V = 232             # ...and it has to be bright in absolute terms
SPEC_MAX_S = 48              # ...and desaturated (a reflection, not ink)
FIELD_GLARE_PAGE_MAX = 248   # a page whose own paper sits above this is simply
                             # over-exposed; no region can be "elevated"
FIELD_GLARE_MARGIN = 6       # how far the field must rise above the paper level
CLIPPED_L = 250              # luminance considered clipped


class LightingReport(dict):
    """A plain dict; subclassed only so it prints readably."""

    def __str__(self) -> str:  # pragma: no cover - convenience only
        return json.dumps(self, indent=2, default=_jsonable)


def _jsonable(o: Any):
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


# ----------------------------------------------------------------------------
# Small helpers
# ----------------------------------------------------------------------------

def _is_gray(img: np.ndarray) -> bool:
    return img.ndim == 2 or (img.ndim == 3 and img.shape[2] == 1)


def _luma(img: np.ndarray) -> np.ndarray:
    """Luminance as float32 0..255."""
    if _is_gray(img):
        return img.reshape(img.shape[0], img.shape[1]).astype(np.float32)
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)


def _downscale(img: np.ndarray, max_dim: int = ANALYSIS_MAX_DIM) -> np.ndarray:
    h, w = img.shape[:2]
    m = max(h, w)
    if m <= max_dim:
        return img
    s = max_dim / float(m)
    return cv2.resize(img, (max(1, int(round(w * s))), max(1, int(round(h * s)))),
                      interpolation=cv2.INTER_AREA)


def _odd(n: int) -> int:
    n = int(n)
    return n if n % 2 == 1 else n + 1


def _noise_sigma(gray: np.ndarray) -> float:
    """
    Immerkaer's fast noise estimate, median variant.

    The textbook version averages |conv|, which text edges inflate badly - a
    sharp receipt reads as a noisy one. Taking the median instead makes the
    estimate robust to the small fraction of pixels that are strokes.
    """
    h, w = gray.shape[:2]
    if h < 5 or w < 5:
        return 0.0
    m = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float32)
    conv = np.abs(cv2.filter2D(gray.astype(np.float32), -1, m))[2:-2, 2:-2]
    # For pure Gaussian noise, median|conv| = 4.047 * sigma.
    return float(np.median(conv) / 4.047)


def _polarity(gray_small: np.ndarray) -> str:
    """
    Guess 'dark_ink' (dark text on light paper) or 'light_ink' (chalkboard).

    Histogram-tail tests fail badly here: faint handwriting contributes a tiny
    dark tail while a single glare streak contributes a huge bright one, so a
    glossy whiteboard reads as white-on-black and the whole correction inverts.

    Instead compare thin-structure energy. Black-hat (closing - image) responds
    to thin DARK features, top-hat (image - opening) to thin BRIGHT ones. Large
    smooth glare blobs are wider than the kernel, so they are absorbed into the
    morphological background and do not pollute either residual.
    """
    h, w = gray_small.shape[:2]
    k = _odd(max(7, int(min(h, w) * 0.02)))
    ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    g = cv2.GaussianBlur(gray_small.astype(np.float32), (0, 0), 1.0)
    blackhat = cv2.morphologyEx(g, cv2.MORPH_BLACKHAT, ker)
    tophat = cv2.morphologyEx(g, cv2.MORPH_TOPHAT, ker)
    # Compare the strong responses only; low-level texture is present in both.
    dark = float(blackhat[blackhat > np.percentile(blackhat, 99)].sum())
    light = float(tophat[tophat > np.percentile(tophat, 99)].sum())
    return "light_ink" if light > dark * 1.6 else "dark_ink"


def _background_field(gray_small: np.ndarray, polarity: str) -> np.ndarray:
    """
    Estimate the smooth illumination field: the image with the ink removed.

    Morphological closing with a kernel wider than the thickest stroke erases
    dark text and leaves the paper; a blur then smooths the residue. For
    light-on-dark we open instead of close.
    """
    h, w = gray_small.shape[:2]
    k = _odd(max(BG_KERNEL_MIN, int(min(h, w) * BG_KERNEL_FRAC)))
    ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    op = cv2.MORPH_CLOSE if polarity == "dark_ink" else cv2.MORPH_OPEN
    # Pre-smooth first: closing dilates, so on grainy paper it latches onto the
    # brightest speck under the kernel and biases the whole field upward by
    # several levels - enough to report a clean page as over-exposed.
    src = cv2.GaussianBlur(gray_small.astype(np.float32), (0, 0), 1.2)
    bg = cv2.morphologyEx(src, op, ker)
    bg = cv2.GaussianBlur(bg, (_odd(k), _odd(k)), 0)
    return np.maximum(bg, 1.0)


def _specular_masks(img: np.ndarray,
                    gray: np.ndarray,
                    bg: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Returns (specular, clipped, unrecoverable) as bool arrays.

    A highlight is defined RELATIVE to the paper level, never absolutely:
    white paper is bright and desaturated too, so an absolute threshold flags
    every clean receipt as wall-to-wall glare.

    Two detectors, because they fail in opposite situations:

      local  - pixel brighter than its own local background. Catches small,
               sharp highlights.
      field  - the background FIELD itself sitting far above the page's paper
               level. Catches large, smooth blowouts, which the local test
               misses entirely: a big soft blob gets absorbed into the
               background estimate, so nothing ever looks brighter than it.

    unrecoverable = specular AND clipped: the surface mirrored the lamp into
    the sensor and whatever stroke was there is simply not in the file.
    """
    clipped = gray >= CLIPPED_L
    paper_ref = float(np.median(bg))

    local_hot = (gray >= bg + SPEC_ABOVE_BG) & (gray >= SPEC_MIN_V)

    # The field test only has meaning while the page as a whole is NOT near
    # clipping. If the whole sheet is pinned at 254 there is no "elevated"
    # region to find - that is global over-exposure, which `bright` covers.
    if paper_ref < FIELD_GLARE_PAGE_MAX:
        field_hot = (bg >= max(paper_ref + FIELD_GLARE_MARGIN,
                               FIELD_GLARE_PAGE_MAX)) & (gray >= SPEC_MIN_V)
    else:
        field_hot = np.zeros_like(local_hot)

    hot = local_hot | field_hot

    if _is_gray(img):
        spec = hot
    else:
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        spec = hot & (hsv[:, :, 1] <= SPEC_MAX_S)
    return spec, clipped, (spec & clipped)


def _blown_over_text(gray: np.ndarray, bg: np.ndarray, lost: np.ndarray) -> float:
    """
    Percentage of the blown-out area that falls inside the region where writing
    was found. Glare on an empty corner is harmless; the same glare across the
    text is the difference between a usable photo and a retake.

    Text under a blowout has already been erased, so it cannot be detected
    directly - we use the bounding box of the ink that IS still visible as a
    stand-in for "where the writing lives".
    """
    if lost.sum() == 0:
        return 0.0
    depth = bg - gray
    thr = max(6.0, float(np.percentile(depth, 99.5)) * 0.30)
    ink = (depth > thr).astype(np.uint8)
    if ink.sum() < 20:
        return 0.0
    ys, xs = np.nonzero(ink)
    y0, y1 = int(np.percentile(ys, 1)), int(np.percentile(ys, 99))
    x0, x1 = int(np.percentile(xs, 1)), int(np.percentile(xs, 99))
    box = np.zeros_like(ink, dtype=bool)
    box[y0:y1 + 1, x0:x1 + 1] = True
    return float((lost & box).sum() / max(1, lost.sum()) * 100.0)


def _ink_stats(gray: np.ndarray, bg: np.ndarray) -> Tuple[float, float]:
    """
    Mean ink depth (how far strokes sit below the local background) and the
    fraction of pixels that look like ink. Measured against the illumination
    field, so a dim photo of good handwriting doesn't read as 'no text'.
    """
    depth = bg - gray
    thr = max(6.0, float(np.percentile(depth, 99.5)) * 0.30)
    ink = depth > thr
    frac = float(ink.mean())
    if frac < 1e-5:
        return 0.0, frac
    return float(depth[ink].mean()), frac


# ----------------------------------------------------------------------------
# analyze()
# ----------------------------------------------------------------------------

def analyze(img: np.ndarray, polarity: str = "dark_ink") -> LightingReport:
    """
    Measure the lighting of `img` without modifying it.

    polarity : 'dark_ink' (default) | 'light_ink' | 'auto'
        Nearly every real input - receipts, paper, signs, whiteboards - is dark
        marks on a light surface, so that is the default. 'auto' guesses, and
        is only worth using if chalkboards are genuinely in scope: guessing
        wrong inverts the entire correction.

    Returns a dict with raw metrics, a list of `issues`, a 0..1 `severity`,
    and a `recommend` block the dispatcher can act on.
    """
    if img is None or img.size == 0:
        raise ValueError("analyze() got an empty image")
    if polarity not in ("dark_ink", "light_ink", "auto"):
        raise ValueError(f"unknown polarity {polarity!r}")

    small = _downscale(img)
    gray_s = _luma(small)

    p1, p5, p50, p95, p99 = np.percentile(gray_s, [1, 5, 50, 95, 99])
    mean = float(gray_s.mean())

    pol = _polarity(gray_s) if polarity == "auto" else polarity
    bg = _background_field(gray_s, pol)

    bg_p5, bg_p50, bg_p95 = np.percentile(bg, [5, 50, 95])
    illum_spread = float(bg_p95 - bg_p5)
    illum_ratio = illum_spread / max(1.0, float(bg_p50))

    spec_s, clip_s, lost_s = _specular_masks(small, gray_s, bg)
    spec_frac = float(spec_s.mean())
    clip_frac = float(clip_s.mean())
    lost_frac = float(lost_s.mean())
    dark_frac = float((gray_s <= 5).mean())

    ink_depth, ink_frac = _ink_stats(gray_s, bg)
    sigma = _noise_sigma(gray_s)

    # Exposure is judged on the PAPER level, not the overall median. A clean
    # receipt legitimately has a median near 248; that is not over-exposure.
    issues = []
    if bg_p50 < DIM_BG:
        issues.append("dim")
    if bg_p50 > BRIGHT_BG:
        issues.append("bright")
    if illum_ratio > UNEVEN_RATIO:
        issues.append("uneven")
    if spec_frac > GLARE_FRAC:
        issues.append("glare")
    if ink_frac > 1e-4 and ink_depth < LOW_INK_CONTRAST:
        issues.append("low_contrast")
    if sigma > NOISE_SIGMA_DENOISE:
        issues.append("noisy")

    parts = [
        min(1.0, max(0.0, (DIM_BG - bg_p50) / DIM_BG)),
        min(1.0, max(0.0, (bg_p50 - BRIGHT_BG) / (255.0 - BRIGHT_BG))),
        min(1.0, illum_ratio / 0.45),
        min(1.0, spec_frac / 0.08),
        min(1.0, max(0.0, (LOW_INK_CONTRAST - ink_depth) / LOW_INK_CONTRAST)),
    ]
    parts = [float(p) for p in parts]
    severity = float(min(1.0, max(parts) * 0.75 + (sum(parts) - max(parts)) * 0.12))

    return LightingReport({
        "issues": issues,
        "severity": round(severity, 3),
        "polarity": pol,
        "metrics": {
            "mean": round(mean, 2),
            "p1": round(float(p1), 2),
            "p5": round(float(p5), 2),
            "p50": round(float(p50), 2),
            "p95": round(float(p95), 2),
            "p99": round(float(p99), 2),
            "paper_level": round(float(bg_p50), 2),
            "paper_low": round(float(bg_p5), 2),
            "paper_high": round(float(bg_p95), 2),
            "illum_spread": round(illum_spread, 2),
            "illum_ratio": round(illum_ratio, 4),
            "specular_frac": round(spec_frac, 5),
            "clipped_frac": round(clip_frac, 5),
            "unrecoverable_frac": round(lost_frac, 5),
            "black_frac": round(dark_frac, 5),
            "ink_depth": round(ink_depth, 2),
            "ink_frac": round(ink_frac, 5),
            "noise_sigma": round(sigma, 2),
        },
        "recommend": {
            "needs_fix": bool(issues),
            "strength": "strong" if severity > 0.25 else "light",
            "denoise": bool(sigma > NOISE_SIGMA_DENOISE),
            "unrecoverable_pct": round(lost_frac * 100.0, 2),
            "retake_advised": bool(lost_frac > 0.02),
        },
        "shape": {"h": int(img.shape[0]), "w": int(img.shape[1]),
                  "gray": bool(_is_gray(img))},
    })


# ----------------------------------------------------------------------------
# fix()
# ----------------------------------------------------------------------------

def _denoise(img: np.ndarray, sigma: float) -> np.ndarray:
    """
    Edge-preserving denoise sized to the measured noise. Runs BEFORE any lift,
    because brightening a dark frame brightens its grain by the same factor and
    OCR reads that grain as speckle.
    """
    h = float(np.clip(sigma * 1.6, 3, 18))
    if img.shape[0] * img.shape[1] <= NLM_MAX_PIXELS:
        if _is_gray(img):
            return cv2.fastNlMeansDenoising(
                img.reshape(img.shape[0], img.shape[1]), None, h, 7, 21)
        return cv2.fastNlMeansDenoisingColored(img, None, h, h, 7, 21)
    # Full-res phone frames: NL-means is too slow, bilateral is close enough.
    return cv2.bilateralFilter(img, 7, float(np.clip(sigma * 4, 15, 90)), 9)


def _white_balance(img: np.ndarray, ref_mask: np.ndarray) -> np.ndarray:
    """
    Neutralise a colour cast using the paper itself as the white reference.
    Documents hand us something most scenes don't: a large region we know is
    meant to be neutral.
    """
    if _is_gray(img) or ref_mask.sum() < 50:
        return img
    f = img.astype(np.float32)
    means = [float(f[:, :, c][ref_mask].mean()) for c in range(3)]
    g = float(np.mean(means))
    out = f.copy()
    for c in range(3):
        if means[c] > 1.0:
            out[:, :, c] = f[:, :, c] * float(np.clip(g / means[c], 0.7, 1.4))
    return np.clip(out, 0, 255).astype(np.uint8)


def _apply_gain(img: np.ndarray, gain: np.ndarray) -> np.ndarray:
    """
    Apply a per-pixel luminance gain, keeping colour ratios. Scaling RGB beats
    a LAB round-trip here: same luminance result, no desaturation.
    """
    if _is_gray(img):
        base = img.reshape(img.shape[0], img.shape[1]).astype(np.float32)
        return np.clip(base * gain, 0, 255).astype(np.uint8)
    return np.clip(img.astype(np.float32) * gain[:, :, None], 0, 255).astype(np.uint8)


def fix(img: np.ndarray,
        strength: str = "auto",
        white_balance: bool = True,
        denoise: str | bool = "auto",
        polarity: str = "dark_ink",
        report: Dict[str, Any] | None = None) -> Tuple[np.ndarray, LightingReport]:
    """
    Normalise the lighting of `img`.

    strength      : 'auto' | 'light' | 'strong'  (see module docstring)
    white_balance : neutralise colour cast using the paper as white reference
    denoise       : 'auto' | True | False
    polarity      : 'dark_ink' (default) | 'light_ink' | 'auto'

    Returns (image, report). Same dtype and channel count as the input, never
    binarised.
    """
    if img is None or img.size == 0:
        raise ValueError("fix() got an empty image")
    if strength not in ("auto", "light", "strong"):
        raise ValueError(f"unknown strength {strength!r}")

    rep = LightingReport(dict(report) if report else dict(analyze(img, polarity)))
    sev = float(rep["severity"])
    mode = "strong" if strength == "auto" else strength
    pol = rep["polarity"]
    actions = []
    work = img.copy()

    # --- 1. denoise before any lift -----------------------------------------
    want_dn = (denoise is True) or (denoise == "auto" and rep["recommend"]["denoise"])
    if want_dn:
        sigma = float(rep["metrics"]["noise_sigma"])
        work = _denoise(work, sigma)
        actions.append(f"denoise(sigma={sigma:.1f})")

    gray_full = _luma(work)
    small = _downscale(work)
    gray_s = _luma(small)

    # --- 2. illumination field: estimated small, applied full-res ------------
    bg_s = _background_field(gray_s, pol)
    spec_s, clip_s, lost_s = _specular_masks(small, gray_s, bg_s)

    # A blown highlight must not drag the field to 255 and crush the paper
    # around it, so cap the field at a high percentile of itself.
    cap = float(np.percentile(bg_s, 99.0))
    bg_s = np.minimum(bg_s, max(cap, 1.0))

    bg = cv2.resize(np.maximum(bg_s, 1.0), (work.shape[1], work.shape[0]),
                    interpolation=cv2.INTER_LINEAR)
    bg = np.maximum(bg, 1.0)

    spec_full = cv2.resize(spec_s.astype(np.uint8),
                           (work.shape[1], work.shape[0]),
                           interpolation=cv2.INTER_NEAREST).astype(bool)
    clip_full = gray_full >= CLIPPED_L
    lost_full = spec_full & clip_full
    lost_pct = float(lost_full.mean() * 100.0)

    # How much of the blown area sits where the writing is? Glare on an empty
    # corner of a whiteboard costs nothing; the same glare across the text is
    # the difference between a usable photo and a retake.
    risk_pct = _blown_over_text(gray_s, bg_s, (spec_s & clip_s))

    # Flat-field: divide the illumination out.
    if pol == "dark_ink":
        gain_flat = TARGET_PAPER / bg
    else:
        gain_flat = (255.0 - TARGET_PAPER) / np.maximum(255.0 - bg, 1.0)

    # 'light' mode corrects only partially, so the page-vs-background gradient
    # that corner detection relies on is not flattened into nothing.
    alpha = 1.0 if mode == "strong" else float(np.clip(0.35 + 0.35 * sev, 0.3, 0.7))
    gain = np.clip(1.0 + (gain_flat - 1.0) * alpha, MIN_GAIN, MAX_GAIN).astype(np.float32)

    work = _apply_gain(work, gain)
    actions.append(f"flat_field(alpha={alpha:.2f},target={TARGET_PAPER:.0f},"
                   f"max_gain={float(gain.max()):.1f})")

    # --- 3. white balance off the paper --------------------------------------
    if white_balance and not _is_gray(work):
        g2 = _luma(work)
        ref = (g2 > np.percentile(g2, 70)) & (~spec_full)  # highlights are not
        work = _white_balance(work, ref)                   # a colour reference
        actions.append("white_balance(paper_ref)")

    # --- 4. exposure trim ----------------------------------------------------
    # After a full flat-field this is nearly a no-op; it earns its keep in
    # 'light' mode, where only part of the field was divided out.
    g3 = _luma(work)
    bg3 = _background_field(_downscale(g3.astype(np.uint8)), pol)
    paper = float(np.median(bg3))
    if paper > 2.0 and abs(paper - TARGET_PAPER) > 6.0:
        g = float(np.clip(np.log(TARGET_PAPER / 255.0) /
                          np.log(np.clip(paper, 2.0, 254.0) / 255.0), 0.4, 2.5))
        lut = np.clip(((np.arange(256) / 255.0) ** g) * 255.0, 0, 255).astype(np.uint8)
        work = cv2.LUT(work, lut)
        actions.append(f"gamma({g:.2f})")

    # --- 5. contrast stretch, anchored to the paper level --------------------
    # Anchored, not stretched to full range. Mapping p99.6 -> 255 looks fine
    # once, but it is not idempotent: each pass shifts the paper higher, so a
    # frame that goes through lighting twice (which the combined pipeline will
    # do - light before geometry, strong after) marches toward a blown page.
    # Pinning ink -> 0 and paper -> TARGET_PAPER gives the same contrast and
    # lands on the same place no matter how many times it runs.
    g4 = _luma(work)
    valid = ~clip_full
    if valid.sum() < g4.size * 0.05:
        valid = np.ones_like(clip_full)
    sample = g4[valid]
    if sample.size > 100:
        lo = float(np.percentile(sample, 0.4))
        paper_now = float(np.median(_background_field(_luma(_downscale(work)), pol)))
        if paper_now - lo > 12.0 and (lo > 3.0 or abs(paper_now - TARGET_PAPER) > 4.0):
            scale = TARGET_PAPER / (paper_now - lo)
            lut = np.clip((np.arange(256, dtype=np.float32) - lo) * scale,
                          0, 255).astype(np.uint8)
            work = cv2.LUT(work, lut)
            actions.append(f"stretch(ink={lo:.0f}->0,paper={paper_now:.0f}->{TARGET_PAPER:.0f})")

    # --- 6. local contrast, only if the page still needs it ------------------
    # Flat-fielding already normalised local brightness, so CLAHE is a top-up,
    # not a staple. Run unconditionally and it rings a bright halo around every
    # stroke and stamps its tile grid across empty paper - both of which OCR
    # sees as structure that is not there.
    small_now = _luma(_downscale(work))
    bg_now = _background_field(small_now, pol)
    depth_now, _ = _ink_stats(small_now, bg_now)
    sigma_now = _noise_sigma(small_now)

    if depth_now < CLAHE_SKIP_INK_DEPTH:
        base = CLAHE_CLIP_STRONG if mode == "strong" else CLAHE_CLIP_LIGHT
        clip_limit = float(np.clip(base * (1.0 - sigma_now / 9.0), 0.6, base))
        # Tiles must be comfortably larger than a glyph, or CLAHE treats the
        # text itself as the local histogram and halos it.
        tile = int(np.clip(min(work.shape[:2]) // 160, 2, CLAHE_TILES))
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile, tile))
        if _is_gray(work):
            work = clahe.apply(work.reshape(work.shape[0], work.shape[1]))
        else:
            lab = cv2.cvtColor(work, cv2.COLOR_BGR2LAB)
            lab[:, :, 0] = clahe.apply(lab[:, :, 0])
            work = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
        actions.append(f"clahe(clip={clip_limit:.2f},tiles={tile})")
    else:
        actions.append(f"clahe(skipped,ink_depth={depth_now:.0f})")

    # --- 7. report -----------------------------------------------------------
    after = analyze(work, pol)
    return work, LightingReport({
        "mode": mode,
        "actions": actions,
        "before": rep,
        "after": after,
        "glare": {
            "unrecoverable_pct": round(lost_pct, 2),
            "blown_over_text_pct": round(risk_pct, 1),
            "note": ("blown-out region(s) present; text there cannot be recovered "
                     "from a single frame - retake advised"
                     if lost_pct > 2.0 else
                     "no significant unrecoverable region"),
            "retake_advised": bool(lost_pct > 2.0),
        },
        "improved": {
            "paper_level": [rep["metrics"]["paper_level"], after["metrics"]["paper_level"]],
            "ink_depth": [rep["metrics"]["ink_depth"], after["metrics"]["ink_depth"]],
            "illum_ratio": [rep["metrics"]["illum_ratio"], after["metrics"]["illum_ratio"]],
            "severity": [rep["severity"], after["severity"]],
        },
    })


# ----------------------------------------------------------------------------
# Debug strip
# ----------------------------------------------------------------------------

def debug_strip(before: np.ndarray, after: np.ndarray, rep: Dict[str, Any]) -> np.ndarray:
    """Before / illumination field / after, side by side."""
    def to_bgr(x):
        return x if not _is_gray(x) else cv2.cvtColor(
            x.reshape(x.shape[0], x.shape[1]), cv2.COLOR_GRAY2BGR)

    b, a = to_bgr(before), to_bgr(after)
    pol = rep.get("before", {}).get("polarity", "dark_ink")
    field = _background_field(_luma(_downscale(before)), pol)
    field = cv2.normalize(field, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    field = cv2.applyColorMap(cv2.resize(field, (before.shape[1], before.shape[0])),
                              cv2.COLORMAP_INFERNO)

    H = 520
    def rs(x):
        s = H / x.shape[0]
        return cv2.resize(x, (max(1, int(x.shape[1] * s)), H))

    panes = [rs(b), rs(field), rs(a)]
    strip = np.hstack(panes)
    x = 0
    for p, lab in zip(panes, ["before", "illumination field", "after"]):
        for col, th in (((0, 0, 0), 4), ((255, 255, 255), 1)):
            cv2.putText(strip, lab, (x + 10, 28), cv2.FONT_HERSHEY_SIMPLEX,
                        0.8, col, th, cv2.LINE_AA)
        x += p.shape[1]
    return strip


# ----------------------------------------------------------------------------
# CLI
# ----------------------------------------------------------------------------

def _read(path: str) -> np.ndarray:
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise SystemExit(f"could not read image: {path}")
    if img.ndim == 3 and img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
    return img


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description="Fix lighting in a document photo.")
    ap.add_argument("input")
    ap.add_argument("output", nargs="?")
    ap.add_argument("--strength", default="auto", choices=["auto", "light", "strong"])
    ap.add_argument("--analyze", action="store_true", help="print metrics only")
    ap.add_argument("--no-wb", action="store_true", help="skip white balance")
    ap.add_argument("--denoise", default="auto", choices=["auto", "on", "off"])
    ap.add_argument("--polarity", default="dark_ink",
                    choices=["dark_ink", "light_ink", "auto"],
                    help="dark marks on light surface (default), the inverse "
                         "(chalkboard), or guess")
    ap.add_argument("--debug", metavar="PATH", help="write a before/field/after strip")
    a = ap.parse_args(argv)

    img = _read(a.input)

    if a.analyze:
        print(json.dumps(analyze(img, a.polarity), indent=2, default=_jsonable))
        return 0
    if not a.output:
        ap.error("output path required unless --analyze")

    dn = {"auto": "auto", "on": True, "off": False}[a.denoise]
    out, rep = fix(img, strength=a.strength, white_balance=not a.no_wb,
                   denoise=dn, polarity=a.polarity)

    d = os.path.dirname(os.path.abspath(a.output))
    os.makedirs(d, exist_ok=True)
    cv2.imwrite(a.output, out)
    if a.debug:
        cv2.imwrite(a.debug, debug_strip(img, out, rep))

    print(json.dumps(rep, indent=2, default=_jsonable))
    if rep["glare"]["retake_advised"]:
        print(f"\n!! {rep['glare']['unrecoverable_pct']}% of the frame is blown out. "
              f"Text there is gone; a retake is the only fix.", file=sys.stderr)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%team_module rotation_consensus
"""Conservative small-roll correction from independent detected text rows.

No OCR text/confidence is used. A box direction must agree with a robust fit
of glyph-component centres in its pixels. Three spatially independent rows
must agree before a single expanded-canvas rotation is allowed. Ambiguous
cursive/short/faint text is deliberately left unchanged.
"""
from __future__ import annotations

import math
import cv2
import numpy as np

MIN_LINES = 3
MIN_AGREEMENT = 0.70
ANGLE_TOL = 2.0
PIXEL_BOX_TOL = 2.5
MIN_COMPONENTS = 5
MIN_ASPECT = 3.0


def _pixel_direction(image, quad, line_h):
    """Robust centre-to-centre slope, not the slant of a letter's strokes."""
    h, w = image.shape[:2]
    lo = np.floor(quad.min(0) - .15 * line_h).astype(int)
    hi = np.ceil(quad.max(0) + .15 * line_h).astype(int)
    x0, y0 = np.maximum(lo, 0)
    x1, y1 = np.minimum(hi, [w, h])
    if x1 <= x0 or y1 <= y0:
        return None, "empty text region"
    crop = image[y0:y1, x0:x1]
    scale = min(1., 1200 / crop.shape[1], 400 / crop.shape[0])
    if scale < 1:
        crop = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY).astype(np.float32)
    lh = line_h * scale
    background = cv2.GaussianBlur(gray, (0, 0), max(3., lh))
    ink = ((gray / np.maximum(background, 1)) < .80).astype(np.uint8)
    # Restrict evidence to the detector's actual polygon: outside dividers and
    # room edges cannot vote even if they fall inside its axis-aligned bounds.
    region = np.zeros_like(ink)
    cv2.fillConvexPoly(region, np.int32((quad - [x0, y0]) * scale), 1)
    ink &= region
    _, _, stats, centres = cv2.connectedComponentsWithStats(ink)
    valid = [i for i, (x, y, cw, ch, area) in enumerate(stats[1:], 1)
             if area >= max(3, .015 * lh * lh) and .18 * lh <= ch <= 1.6 * lh
             and .04 * lh <= cw <= 2.5 * lh]
    points = centres[valid]
    if len(points) < MIN_COMPONENTS:
        return None, "too few independent glyph components"
    span = float(np.ptp(points[:, 0]))
    if span < .55 * (x1 - x0) * scale:
        return None, "glyph evidence covers too little of the line"
    slopes = [(points[j, 1] - points[i, 1]) / (points[j, 0] - points[i, 0])
              for i in range(len(points)) for j in range(i + 1, len(points))
              if abs(points[j, 0] - points[i, 0]) >= .25 * span]
    if not slopes:
        return None, "insufficient component separation"
    slope = float(np.median(slopes))
    intercept = float(np.median(points[:, 1] - slope * points[:, 0]))
    residual = float(np.median(np.abs(points[:, 1] - slope * points[:, 0] - intercept)))
    if residual > .25 * lh:
        return None, "components do not form a straight text row"
    return float(np.degrees(np.arctan(slope))), "pixel baseline supported"


def estimate(image, boxes, min_angle=2., max_angle=15.):
    """Return an auditable decision; confidence is agreement, NOT a probability."""
    info = dict(angle=0., confidence=0., used=False, applied=False,
                method="independent-text-row-consensus", reason="insufficient independent text rows",
                candidates=[], independent_rows=0, agreeing_rows=0, stacked_rows=0)
    if boxes is None:
        info["reason"] = "no detected text boxes supplied: abstain"
        return info
    candidates = []
    for index, box in enumerate(boxes):
        try:
            q = np.asarray(box, np.float32).reshape(-1, 2)
        except (ValueError, TypeError):
            info["reason"] = "invalid detected geometry: abstain"
            return info
        if q.shape != (4, 2) or not np.isfinite(q).all():
            info["reason"] = "invalid detected geometry: abstain"
            return info
        direction = ((q[1] - q[0]) + (q[2] - q[3])) / 2
        if direction[0] < 0:
            direction = -direction
        width = float(np.linalg.norm(direction))
        height = float((np.linalg.norm(q[3] - q[0]) + np.linalg.norm(q[2] - q[1])) / 2)
        angle = float(np.degrees(np.arctan2(direction[1], direction[0])))
        if height < 8 or width < 80 or width < MIN_ASPECT * height or abs(angle) > 45:
            continue
        pixel, reason = _pixel_direction(image, q, height)
        # Keep large/conflicting horizontal rows in the denominator. Dropping
        # them would let a small tilted minority manufacture a consensus.
        small_roll = abs(angle) <= max_angle
        agreed = small_roll and pixel is not None and abs(pixel - angle) <= PIXEL_BOX_TOL
        if not small_roll:
            reason = "box angle outside small-roll range"
        elif pixel is not None and not agreed:
            reason = "pixel/box direction disagreement"
        row = dict(index=index, box_angle=round(angle, 3), pixel_angle=pixel,
                   reliable=bool(agreed), reason=reason,
                   height=height, width=width, centre=q.mean(0).tolist())
        info["candidates"].append(row)
        candidates.append(row)
    if not candidates:
        return info
    # Fragments/duplicate boxes from the SAME physical row must not manufacture
    # a majority. Compare row intercepts under the median box direction.
    reference = float(np.median([r["box_angle"] for r in candidates]))
    slope = math.tan(math.radians(reference))
    groups = []
    for row in sorted(candidates, key=lambda r: -r["width"]):
        intercept = (row["centre"][1] - slope * row["centre"][0]) * math.cos(math.radians(reference))
        if any(abs(intercept - other["intercept"]) < .7 * min(row["height"], other["height"])
               for other in groups):
            continue
        groups.append(dict(row, intercept=intercept))
    info["independent_rows"] = len(groups)
    reliable = [r for r in groups if r["reliable"]]
    if len(groups) < MIN_LINES or len(reliable) < MIN_LINES:
        return info
    clusters = [[r for r in reliable if abs(r["pixel_angle"] - seed["pixel_angle"]) <= ANGLE_TOL]
                for seed in reliable]
    inliers = max(clusters, key=len)
    angle = float(np.median([r["pixel_angle"] for r in inliers]))
    # Re-evaluate around the final estimate, not just the winning seed.
    inliers = [r for r in reliable if abs(r["pixel_angle"] - angle) <= ANGLE_TOL]
    agreement = len(inliers) / len(groups)
    info.update(angle=round(angle, 3), confidence=round(agreement, 3), agreeing_rows=len(inliers))
    if len(inliers) < MIN_LINES or agreement < MIN_AGREEMENT:
        info["reason"] = "independent rows disagree: abstain"
        return info
    if np.ptp([r["intercept"] for r in inliers]) < 2 * np.median([r["height"] for r in inliers]):
        info["reason"] = "insufficient separation between text rows: abstain"
        return info
    # Three short captions across the board must not masquerade as three
    # stacked paragraph rows merely because their handwritten baselines differ.
    # Require a common horizontal span within at least one column of writing.
    # Use the baseline axis so this check is invariant to small camera roll.
    radians = math.radians(angle)
    axis = np.array([math.cos(radians), math.sin(radians)])
    intervals = [(float(np.dot(r["centre"], axis)) - .375 * r["width"],
                  float(np.dot(r["centre"], axis)) + .375 * r["width"], r) for r in inliers]
    stacked = max(([r for lo, hi, r in intervals if lo <= pivot <= hi]
                   for pivot in [v for lo, hi, _ in intervals for v in (lo, hi)]), key=len)
    info["stacked_rows"] = len(stacked)
    if (len(stacked) < MIN_LINES or
            np.ptp([r["intercept"] for r in stacked]) < 2 * np.median([r["height"] for r in stacked])):
        info["reason"] = "no column with enough separated text rows: abstain"
        return info
    if not min_angle <= abs(angle) <= max_angle:
        info["reason"] = "tilt below trigger or outside small-roll range: unchanged"
        return info
    info.update(used=True, reason="independent text rows and pixel baselines agree")
    return info


def correct(image, boxes, min_angle=2., max_angle=15.):
    info = estimate(image, boxes, min_angle, max_angle)
    if not info["used"]:
        return image, info
    h, w = image.shape[:2]
    matrix = cv2.getRotationMatrix2D((w / 2, h / 2), info["angle"], 1.)
    corners = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    projected = cv2.transform(corners[None], matrix)[0]
    lo, hi = np.floor(projected.min(0)) - 1, np.ceil(projected.max(0)) + 1
    matrix[:, 2] -= lo
    nw, nh = np.int32(hi - lo)
    text = np.vstack([np.asarray(b, np.float32).reshape(4, 2) for b in boxes])
    mapped = cv2.transform(text[None], matrix)[0]
    if not np.all((mapped[:, 0] >= 0) & (mapped[:, 0] < nw) &
                  (mapped[:, 1] >= 0) & (mapped[:, 1] < nh)):
        info.update(used=False, reason="rotation would clip detected writing: unchanged")
        return image, info
    border = np.median(np.vstack([image[0], image[-1], image[:, 0], image[:, -1]]), axis=0)
    rotated = cv2.warpAffine(image, matrix, (int(nw), int(nh)), flags=cv2.INTER_CUBIC,
                             borderMode=cv2.BORDER_CONSTANT, borderValue=tuple(float(v) for v in border))
    info.update(applied=True, transform=matrix.tolist(), output_shape=[int(nh), int(nw)])
    return rotated, info

In [ ]:
%%team_module line_grouping
"""Join detector fragments in text coordinates, preserving separate rows.

Axis-aligned bounding rectangles overlap vertically for long tilted sentences
even when their actual rows are well separated. Compare distances normal to
the writing direction instead. No OCR strings or labels enter this function.
"""
from __future__ import annotations
import cv2
import numpy as np
from . import text_lines as T

_legacy_group = T.group_into_lines


def group_into_lines(polys,gap_factor=1.2,min_v_overlap=.5):
    boxes=[T.order_tl_tr_br_bl(p) for p in polys]
    directions=[]
    for p in boxes:
        d=(p[1]-p[0]+p[2]-p[3]).astype(float)
        if d[0]<0:d=-d
        directions.append(d/max(float(np.linalg.norm(d)),1e-6))
    heights=[T.line_height(p) for p in boxes]
    groups=[[i] for i in range(len(boxes))]

    def same_row(a,b):
        if min(heights[a],heights[b])<=0 or max(heights[a],heights[b])>2*min(heights[a],heights[b]):return False
        if np.dot(directions[a],directions[b])<np.cos(np.radians(5)):return False
        axis=directions[a]+directions[b];axis/=max(float(np.linalg.norm(axis)),1e-6)
        normal=np.array([-axis[1],axis[0]])
        av,bv=boxes[a]@normal,boxes[b]@normal
        overlap=min(av.max(),bv.max())-max(av.min(),bv.min())
        if overlap<min_v_overlap*min(float(np.ptp(av)),float(np.ptp(bv))):return False
        au,bu=boxes[a]@axis,boxes[b]@axis
        gap=max(bu.min()-au.max(),au.min()-bu.max())
        return gap<gap_factor*max(heights[a],heights[b])

    edges=[(a,b) for a in range(len(boxes)) for b in range(a+1,len(boxes)) if same_row(a,b)]
    # A chain of slightly displaced fragments must not connect distinct rows.
    for a,b in edges:
        ga=next(g for g in groups if a in g);gb=next(g for g in groups if b in g)
        if ga is gb:continue
        joined=ga+gb;axis=np.mean([directions[i] for i in joined],axis=0)
        axis/=max(float(np.linalg.norm(axis)),1e-6);normal=np.array([-axis[1],axis[0]])
        offsets=[float(boxes[i].mean(0)@normal) for i in joined]
        if np.ptp(offsets)>(1-min_v_overlap)*min(heights[i] for i in joined):continue
        ga.extend(gb);groups.remove(gb)
    output=[]
    for group in groups:
        p=boxes[group[0]] if len(group)==1 else T.order_tl_tr_br_bl(cv2.boxPoints(cv2.minAreaRect(np.vstack([boxes[i] for i in group]))))
        output.append((p,sorted(group)))
    return [p for p,_ in output],[g for _,g in output]


def group_with_row_guard(polys,gap_factor=1.2,min_v_overlap=.5):
    """Preserve old fragment joins unless a group spans distinct text rows.

    The first candidate unnecessarily split some useful word fragments. This
    guard requires overlapping along-line spans AND separated baselines
    before replacing an old group. It is a geometric check, not an OCR vote.
    """
    boxes=[T.order_tl_tr_br_bl(p) for p in polys]
    old,groups=_legacy_group(boxes,gap_factor=gap_factor,min_v_overlap=min_v_overlap)
    output=[];members=[]
    for old_box,group in zip(old,groups):
        separated=False
        for k,a in enumerate(group):
            for b in group[k+1:]:
                qa,qb=boxes[a],boxes[b]
                da=(qa[1]-qa[0]+qa[2]-qa[3]).astype(float);da/=max(float(np.linalg.norm(da)),1e-6)
                db=(qb[1]-qb[0]+qb[2]-qb[3]).astype(float);db/=max(float(np.linalg.norm(db)),1e-6)
                if np.dot(da,db)<np.cos(np.radians(12)):continue
                axis=da+db;axis/=max(float(np.linalg.norm(axis)),1e-6)
                normal=np.array([-axis[1],axis[0]])
                au,bu=qa@axis,qb@axis
                overlap=min(au.max(),bu.max())-max(au.min(),bu.min())
                separation=abs(float((qa.mean(0)-qb.mean(0))@normal))
                if (overlap>.3*min(float(np.ptp(au)),float(np.ptp(bu))) and
                        separation>.65*min(T.line_height(qa),T.line_height(qb))):
                    separated=True
        if not separated:
            output.append(old_box);members.append(group)
        else:
            pieces,indices=group_into_lines([boxes[i] for i in group],gap_factor,min_v_overlap)
            output.extend(pieces);members.extend([[group[i] for i in subset] for subset in indices])
    return output,members

In [ ]:
%%team_module pipeline
"""Embedded single-step preprocessing: same implementation as the audit."""
from __future__ import annotations
import time
from dataclasses import dataclass, asdict
import cv2
import numpy as np
from . import board_geometry as geo, text_lines as tl
from . import fix_lighting, rotation_consensus, line_grouping

MAX_ROLL = 15.0

REGION_MARGIN = 0.6

DIRECTIONAL_GROUPING = False

@dataclass(frozen=True)
class Steps:
    lighting_light: bool = False
    perspective: bool = False
    rotation: bool = False
    lighting_strong: bool = False
    blemish: bool = False
    crop_region: bool = True      # cut out the text region before step 5 (needed for rotation/strong lighting)
    unclip: float = 2.0           # detector box growth; 2.0 = the baseline preset (app); smaller keeps lines apart
    min_board_angle: float = 0.0  # perspective is applied only if the board is turned at least this much (deg)
    auto: bool = False            # decide per photo which steps to run (Part 1 checks); the step flags above are
                                  # then ignored
    safety_net: bool = True       # auto only: keep the corrected version only if it keeps the text and the reader is
                                  # surer of it (Part 2)
    auto_blemish: bool = False    # auto only: allow blemish removal (off in the first test: it hurt in round 1 and its
                                  # check did not pick out the blemish photos; the check is still measured and logged)
    isolated: str = ""            # round 4: exactly one optional operation, or 'baseline'

    def name(self):
        if self.isolated:
            return f"isolated {self.isolated}, unclip {self.unclip}"
        if self.auto:
            return (f"auto: per-photo checks{' + safety net' if self.safety_net else ''}"
                    f"{' (blemish allowed)' if self.auto_blemish else ''}, unclip {self.unclip}")
        on = [k for k, v in asdict(self).items()
              if v is True and k not in ("crop_region", "safety_net", "auto_blemish")]
        if self.unclip != 2.0:
            on.append(f"unclip {self.unclip}")
        if self.perspective and self.min_board_angle:
            on.append(f"perspective only if board angle >= {self.min_board_angle:g}")
        return " + ".join(on) if on else "none"

PERSP_MIN_ANGLE = 9.0

PERSP_MIN_TEXT_SCALE = 0.9

LIGHT_ISSUES = ("dim", "low_contrast")

LIGHT_MIN_ILLUM = 0.45

ROT_MIN_ANGLE = 2.0

def _gated_rotation(img, boxes=None):
    # No global projection fallback: board rulings/room edges cannot vote.
    # "confidence" in this result is a row-agreement fraction, not OCR certainty.
    return rotation_consensus.correct(img, boxes, min_angle=ROT_MIN_ANGLE, max_angle=MAX_ROLL)

def _ordered_lines(boxes):
    if DIRECTIONAL_GROUPING:
        lines, _ = line_grouping.group_into_lines(boxes, gap_factor=getattr(tl, 'GROUP_GAP', 1.2))
    else:
        lines, _ = tl.group_into_lines(boxes)
    return [lines[i] for i in tl.reading_order(lines)]

def preprocess_isolated(img_bgr, detector, steps, f35=None):
    """Round 4: a shared ROI policy plus at most ONE gated preprocessing step.

    Gates are provisional geometry/photometry heuristics, not OCR-confidence
    selection. Skipped corrections retain exactly the baseline image path.
    Region detection at native resolution is common to every arm, not counted
    as enlargement. Ordinary quadrilateral line cropping remains common too.
    """
    t0 = time.time()
    mode = steps.isolated
    if mode not in {"baseline", "perspective", "lighting", "rotation", "enlargement"}:
        raise ValueError(f"Unknown isolated step: {mode}")
    boxes, _ = detector.detect(img_bgr, board_mask_fn=geo.segment_board_floodfill, unclip=steps.unclip)
    log = dict(steps=steps.name(), isolated=mode, applied=False,
               reason="baseline: no optional correction", signals={}, rectified=False)
    if not boxes:
        log.update(reason="no initial text: no optional correction", boxes=0, lines=0,
                   seconds=round(time.time() - t0, 3))
        return dict(image=img_bgr, read_image=img_bgr, lines=[], log=log, enlarge_method="none")
    x, pts = img_bgr, np.vstack(boxes)
    if mode == "perspective":
        r = geo.auto_rectify(img_bgr, boxes, f35=f35, proposal_search=True)
        angle = float(r["angle"])
        log["signals"]["board_angle"] = angle if np.isfinite(angle) else None
        log["signals"]["side_view_gate"] = r.get("side_view_gate")
        log["signals"]["proposal_count"] = r.get("proposal_count", 0)
        log["signals"]["proposal_rejections"] = r.get("rejections", [])
        log["signals"]["excluded_box_indices"] = r.get("excluded_indices", [])
        log["signals"]["readability_gate"] = r.get("readability_gate")
        log["signals"]["initial_boxes"] = [np.asarray(b).tolist() for b in boxes]
        if r.get("choice") is not None:
            choice = r["choice"]
            log["signals"]["panel_polygon"] = np.asarray(choice["polygon"]).tolist()
            log["signals"]["frame_segments"] = [[np.asarray(e["p"]).tolist(),np.asarray(e["q"]).tolist()]
                                                   for e in (*choice["pair"],choice["ref"])]
        log["reason"] = r["status"]
        if r["rectified"] is not None and np.isfinite(angle):
            scale = float(np.median([
                tl.line_height(cv2.perspectiveTransform(np.float32(b)[None], r["H"])[0]) /
                max(tl.line_height(b), 1) for b in boxes]))
            log["signals"]["text_scale"] = scale
            preserved = r.get("readability_gate", {}).get("approved", scale >= PERSP_MIN_TEXT_SCALE)
            if angle >= PERSP_MIN_ANGLE and preserved:
                x, pts = r["rectified"], r["text_rect"]
                log.update(applied=True, rectified=True, reason="supported board frame and ink-height checks passed")
            else:
                log["reason"] = "angle or text-size gate failed: original retained"
    # Same writing-region crop policy in ALL arms, including the baseline.
    x0, y0, x1, y1 = geo.crop_box(x.shape, pts, REGION_MARGIN)
    work = x[y0:y1, x0:x1].copy()
    log["region"] = [int(v) for v in (x0, y0, x1, y1)]
    if mode == "lighting":
        rep = fix_lighting.analyze(work)
        issues = [i for i in rep["issues"] if i in LIGHT_ISSUES]
        uneven = float(rep["metrics"]["illum_ratio"])
        log["signals"] = dict(lighting_issues=issues, illum_ratio=uneven)
        if issues or uneven >= LIGHT_MIN_ILLUM:
            work, _ = fix_lighting.fix(work, strength="light")
            log.update(applied=True, reason="lighting gate passed: gentle correction")
        else:
            log["reason"] = "lighting gate not met: original retained"
    elif mode == "rotation":
        rotated, info = _gated_rotation(work, [np.float32(b) - [x0, y0] for b in boxes])
        log["signals"] = info
        if info["used"] and abs(info["angle"]) >= ROT_MIN_ANGLE:
            work = rotated
            log.update(applied=True, reason=info.get("reason", "text-row consensus passed"))
        else:
            log["reason"] = info.get("reason", "rotation gate not met: original retained")
    final_boxes, dinfo = detector.detect(work, board_mask_fn=None, unclip=steps.unclip)
    lines = _ordered_lines(final_boxes)
    if mode == "enlargement":
        n = sum(tl.line_height(p) < tl.REC_MIN_H for p in lines)
        log.update(applied=bool(n), small_lines=int(n),
                   reason=f"bicubic enlargement requested for {n} small line crops")
    log.update(boxes=len(final_boxes), lines=len(lines), detect=dinfo["path"],
               seconds=round(time.time() - t0, 3))
    return dict(image=work, read_image=work, lines=lines, log=log,
                enlarge_method="bicubic" if mode == "enlargement" else "none")

def line_crops(result, prepare=True, method="bicubic+sharpen"):
    """Crops of every line of a preprocess_photo result, plus box heights/widths.
    prepare=True applies the app rule (lines < REC_MIN_H px enlarged to TARGET_H) = what the reader sees."""
    out = []
    method = result.get("enlarge_method", method)
    for p in result["lines"]:
        c = tl.quad_crop(result["read_image"], p)
        h, w = tl.line_height(p), tl.line_width(p)
        out.append(dict(crop=tl.prepare_for_reader(c, h, method) if prepare else c, height=h, width=w, box=p))
    return out

def preprocess_photo(img_bgr, detector, steps, f35=None, reader=None):
    return preprocess_isolated(img_bgr, detector, steps, f35)

In [ ]:
%%team_module team_pipeline
"""Small teammate-facing interface; the research notebook remains separate.

One explicitly selected preprocessing mode is allowed per run. Default:
audited isolated baseline, legacy grouping, native line crops, no SR/blemish.
Detection and recognition models are created once per TeamPipeline instance.
"""
from __future__ import annotations

from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
import hashlib
import json
from pathlib import Path
import re
import shutil
import tempfile
import time
import uuid

import cv2
import numpy as np
from PIL import Image, ImageOps

from . import board_geometry as geometry, line_grouping, pipeline, text_lines

MODES = ('baseline', 'rotation', 'enlargement', 'perspective', 'lighting')
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.heic', '.heif', '.tif', '.tiff', '.bmp', '.webp'}


@dataclass(frozen=True)
class Settings:
    detector_name: str = 'PP-OCRv5_mobile_det'
    recognizer_name: str = 'PP-OCRv6_medium_rec'
    recognizer_dir: str | None = None
    preprocess: str = 'baseline'
    directional_grouping: bool = False
    unclip: float = 1.2

    def validate(self):
        if self.preprocess not in MODES:
            raise ValueError(f'preprocess must be one of {MODES}')
        if not isinstance(self.directional_grouping, bool):
            raise ValueError('directional_grouping must be True or False')
        if not np.isfinite(self.unclip) or self.unclip <= 0:
            raise ValueError('unclip must be positive and finite')
        if self.recognizer_dir is not None:
            folder = Path(self.recognizer_dir).expanduser()
            if not folder.is_dir():
                raise ValueError('recognizer_dir must point to the exported inference MODEL FOLDER')
            required = ['inference.pdiparams', 'inference.yml']
            if not all((folder/name).is_file() for name in required) or not any(
                (folder/name).is_file() for name in ('inference.json', 'inference.pdmodel')):
                raise ValueError('Model folder needs inference.pdiparams, inference.yml and inference.json (or inference.pdmodel)')


def resolve_inputs(entries):
    """Resolve explicit files or directories, with stable ordering and no duplicates."""
    if isinstance(entries, (str, Path)):
        entries = [entries]
    paths = []
    for entry in entries:
        path = Path(entry).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(f'Photo/folder does not exist: {path}')
        candidates = sorted(path.iterdir()) if path.is_dir() else [path]
        selected = [p for p in candidates if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES]
        if not selected:
            raise ValueError(f'No supported photos in: {path}')
        paths.extend(selected)
    unique = list(dict.fromkeys(paths))
    if not unique:
        raise ValueError('Add a photo path or folder to PHOTO_PATHS in the settings cell first.')
    return unique


def load_photo(path):
    """Native pixels with EXIF orientation applied; no preview downscaling."""
    path = Path(path)
    if path.suffix.lower() in {'.heic', '.heif'}:
        try:
            import pillow_heif
        except ImportError as exc:
            raise ImportError('Install pillow-heif to open HEIC/HEIF photos.') from exc
        pillow_heif.register_heif_opener()
    with Image.open(path) as image:
        focal35 = image.getexif().get_ifd(0x8769).get(0xA405)
        rgb = np.array(ImageOps.exif_transpose(image).convert('RGB'))
    focal35 = float(focal35) if focal35 else None
    if focal35 is not None and (not np.isfinite(focal35) or focal35 <= 0):
        focal35 = None
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), focal35


def _json_safe(value):
    if isinstance(value, dict):
        return {str(k): _json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return _json_safe(value.tolist())
    if isinstance(value, np.generic):
        return _json_safe(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


class _RecordingDetector:
    def __init__(self, detector):
        self.detector = detector
        self.last_boxes = []

    def detect(self, *args, **kwargs):
        boxes, info = self.detector.detect(*args, **kwargs)
        self.last_boxes = boxes
        return boxes, info


def _write_image(path, image):
    # imencode + Path works with non-ASCII paths on platforms where imwrite may not.
    ok, encoded = cv2.imencode(path.suffix, image)
    if not ok:
        raise OSError(f'Could not encode {path.name}')
    path.write_bytes(encoded.tobytes())


class TeamPipeline:
    def __init__(self, settings=None, *, detector=None, reader=None):
        self.settings = settings or Settings()
        self.settings.validate()
        # Avoid the macOS mixed PyTorch/Paddle import-order issue when Torch
        # happens to be installed. Torch is not a dependency of this notebook.
        if detector is None or reader is None:
            import importlib.util
            if importlib.util.find_spec('torch') is not None:
                import torch
        self.detector = detector if detector is not None else text_lines.Detector(
            model_name=self.settings.detector_name)
        self.reader = reader if reader is not None else text_lines.Reader(
            model_name=self.settings.recognizer_name,
            model_dir=str(Path(self.settings.recognizer_dir).expanduser().resolve())
            if self.settings.recognizer_dir is not None else None)

    def process_photo(self, photo, output_dir):
        """Save text, JSON, processed/annotated images and the actual OCR crops.

        Exported boxes refer to processed.png, NOT the original photograph.
        Every recognized line is retained; confidence does not certify accuracy.
        """
        started = time.perf_counter()
        source = Path(photo).expanduser().resolve()
        image, focal35 = load_photo(source)
        detector = _RecordingDetector(self.detector)
        result = pipeline.preprocess_photo(image, detector, pipeline.Steps(
            isolated=self.settings.preprocess, unclip=self.settings.unclip,
            safety_net=False), f35=focal35)
        # Reuse the final detection boxes. This makes grouping explicit without
        # mutating pipeline globals or repeating neural-network detection.
        group = line_grouping.group_into_lines if self.settings.directional_grouping else text_lines.group_into_lines
        lines, _ = group(detector.last_boxes)
        result['lines'] = [lines[i] for i in text_lines.reading_order(lines)]
        if self.settings.preprocess == 'enlargement':
            small = sum(text_lines.line_height(p) < text_lines.REC_MIN_H for p in result['lines'])
            result['log'].update(applied=bool(small), small_lines=int(small),
                                 reason=f'bicubic enlargement requested for {small} small line crops')
        result['log']['lines'] = len(result['lines'])
        crops = pipeline.line_crops(result)
        readings = self.reader.read([c['crop'] for c in crops]) if crops else []
        if len(readings) != len(crops):
            raise RuntimeError('Reader returned a different number of results than line crops')

        output = Path(output_dir).expanduser().resolve()
        output.mkdir(parents=True, exist_ok=True)
        stem = re.sub(r'[^A-Za-z0-9_-]+', '_', source.stem)[:70] or 'photo'
        name = f'{stem}_{datetime.now(timezone.utc):%Y%m%dT%H%M%S}_{uuid.uuid4().hex[:8]}'
        target = output/name
        temp = Path(tempfile.mkdtemp(prefix='.writing_', dir=output))
        try:
            (temp/'crops').mkdir()
            annotated = result['read_image'].copy()
            rows = []
            for i, (crop, (text, confidence)) in enumerate(zip(crops, readings), 1):
                crop_file = f'crops/line_{i:03d}.png'
                _write_image(temp/crop_file, crop['crop'])
                q = np.asarray(crop['box'], np.float32)
                cv2.polylines(annotated, [np.int32(q)], True, (0, 190, 0), 2)
                anchor = tuple(np.maximum(q.min(0).astype(int), [0, 18]))
                cv2.putText(annotated, str(i), anchor, cv2.FONT_HERSHEY_SIMPLEX, .65, (0, 0, 220), 2)
                rows.append(dict(line_id=i, text=str(text), confidence=float(confidence),
                    box=q.tolist(), crop=crop_file, source_box_height=float(crop['height']),
                    reader_crop_shape=list(crop['crop'].shape)))
            text = '\n'.join(row['text'] for row in rows)
            versions = {}
            for package in ('paddlepaddle','paddlepaddle-gpu','paddleocr','paddlex','numpy','Pillow'):
                try: versions[package] = version(package)
                except PackageNotFoundError: pass
            data = _json_safe(dict(schema='board2notes.ocr.v1', source=str(source),
                source_sha256=hashlib.sha256(source.read_bytes()).hexdigest(),
                settings=asdict(self.settings), versions=versions,
                original_upright_shape=list(image.shape), processed_shape=list(result['read_image'].shape),
                coordinates='Line boxes are in processed.png pixels; not original photo coordinates.',
                preprocessing=result['log'], line_count=len(rows), text=text, lines=rows,
                seconds=round(time.perf_counter()-started,3)))
            _write_image(temp/'processed.png', result['read_image'])
            _write_image(temp/'annotated.jpg', annotated)
            (temp/'ocr.txt').write_text(text, encoding='utf-8')
            (temp/'ocr.json').write_text(json.dumps(data,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
            temp.rename(target)
        except Exception:
            shutil.rmtree(temp)
            raise
        return dict(output_dir=str(target), text_path=str(target/'ocr.txt'),
                    json_path=str(target/'ocr.json'), annotated_path=str(target/'annotated.jpg'),
                    text=text, line_count=len(rows), decision=data['preprocessing'], seconds=data['seconds'])

In [ ]:
Settings = _embedded_package.team_pipeline.Settings
TeamPipeline = _embedded_package.team_pipeline.TeamPipeline
resolve_inputs = _embedded_package.team_pipeline.resolve_inputs
print("Embedded pipeline ready. No project Python files imported.")

## 2. Set photos, reader and output folder

Add photo paths or a folder below. Relative paths use the working folder printed above. For the team's fine-tuned reader, set **both** its architecture name and exported model folder; otherwise leave `RECOGNIZER_DIR = None` for the named pretrained reader.

`baseline` is the default. Optional single-step modes are `rotation`, `enlargement`, `perspective`, `lighting`. Each uses its own decision rule and may skip a photo. Perspective and lighting remain experimental. Directional grouping also remains experimental and off by default. No SR, sharpening or blemish removal runs.

In [ ]:
PHOTO_PATHS = [
    # Path("/your/path/whiteboard.jpg"),
    # Path("/your/path/folder_of_photos"),
]
OUTPUT_DIR = Path.cwd() / "outputs" / "team_pipeline"

RECOGNIZER_NAME = "PP-OCRv6_medium_rec"
RECOGNIZER_DIR = None
# Fine-tuned example: set BOTH the matching name and your exported folder:
# RECOGNIZER_NAME = "PP-OCRv5_server_rec"
# RECOGNIZER_DIR = Path("/your/path/to/exported_model_folder")

PREPROCESS = "baseline"
DIRECTIONAL_GROUPING = False

settings = Settings(
    recognizer_name=RECOGNIZER_NAME,
    recognizer_dir=str(RECOGNIZER_DIR) if RECOGNIZER_DIR is not None else None,
    preprocess=PREPROCESS,
    directional_grouping=DIRECTIONAL_GROUPING,
)
settings.validate()
photos = resolve_inputs(PHOTO_PATHS)
print(f"{len(photos)} photo(s); selected mode: {settings.preprocess}")
print("Output folder:", OUTPUT_DIR)

## 3. Load models once

Run this again if you change settings or the reader. All photos reuse the same models. A custom reader folder needs `inference.pdiparams`, `inference.yml` and `inference.json` (or the older `inference.pdmodel`). A training checkpoint is not an exported inference folder.

In [ ]:
engine = TeamPipeline(settings)
print("Detector and recognizer are ready.")

## 4. Process photos

Every photo creates a new output folder and retains earlier results. The decision log explains whether the selected correction ran or was skipped.

In [ ]:
results = []
for photo in photos:
    result = engine.process_photo(photo, OUTPUT_DIR)
    results.append(result)
    print(f"{photo.name}: {result['line_count']} lines, {result['seconds']:.1f} s")
    print("Decision:", result["decision"].get("reason", "baseline"))
    print("Saved:", result["output_dir"], "\n")

## 5. Review and hand off to the LLM

Check numbered boxes and the transcript. Each output folder contains:

- `ocr.txt`: raw text in reading order, ready for the LLM stage.
- `ocr.json`: text, line boxes, confidence, settings and preprocessing decisions.
- `processed.png`: exact image used to cut text lines.
- `annotated.jpg`: numbered line boxes.
- `crops/`: actual images fed to the recognizer.

Boxes use **processed.png coordinates**. Confidence is not accuracy; a detector can miss or clip text. CER requires verified labels and is not calculated here.

In [ ]:
from IPython.display import Image as DisplayImage, display

for result in results:
    print(Path(result["output_dir"]).name)
    display(DisplayImage(filename=result["annotated_path"], width=1000))
    print(result["text"] or "[No text detected]")
    print("Text:", result["text_path"])
    print("JSON:", result["json_path"], "\n")

### Reuse after the embedded cells have run

```python
engine = TeamPipeline(Settings())  # create once
result = engine.process_photo("photo.jpg", "outputs/team_pipeline")
raw_text_for_llm = result["text"]
```

This standalone notebook runs inference only and leaves notebook 08, model weights and training data unchanged. The optional ZIP contains this same notebook and setup notes; it contains no Python helper files. You can share the `.ipynb` alone.